# COSC2753 Assignment 2 — Task 1: Fashion Article Type Classification

Predicting `articleType` from a fashion catalogue image: **124 classes**, 37,846 labelled
images at 60x80 pixels, and a class distribution running from 5,267 images down to 1.

Three model families are compared under one protocol. A HOG + linear SVM baseline, a
VGG-style convolutional network trained from scratch, and a small ResNet. Each is tuned on
an equal-sized grid, confirmed at the same epoch budget, refitted on the same rows, and
scored once on a reporting split that no model was tuned against. The selected model then
predicts the 5,829 test images.

**This notebook is self-contained.** It defines every transform, model, training loop and
metric that it uses, trains all three families from scratch in one linear pass, and writes
the finished models out at the end. It imports no project module, reads no checkpoint
written elsewhere, and needs no companion notebook or launcher script.

## How to Run the Notebook

1. **Data.** The notebook needs `preprocessed_datasets/train_manifest.csv`, the audited
   manifest written by `00_eda_and_preprocessing.ipynb`, together with the image folders
   `datasets/train/images_train/` and `datasets/test/images_test/`, and the prediction
   template `datasets/test/styles_prediction.csv`. Section 1.3 finds them by walking up
   from the working directory, so the notebook runs from the repository root and from any
   subdirectory of it alike.
2. **Hardware.** A CUDA GPU is expected. Section 4.5 confirms each neural family under
   both imbalance regimes and Section 5.2 refits each of them, so the run costs about
   one extra confirmation and one extra refit per neural family against the single-arm
   version -- roughly half as long again. Set
   `ALLOW_CPU = True` in Section 1.2 to run without a GPU, which is considerably slower.
3. **Run order.** Top to bottom, once, on a fresh kernel. Nothing is cached between
   sessions and no cell depends on a previous run.
4. **Quick check.** Set `QUICK_RUN = True` in Section 1.2 to exercise the whole path on a
   handful of rows per class in a couple of minutes. The numbers it prints are meaningless;
   it checks that the code runs.
5. **Outputs.** Section 6 writes the refitted models, one per arm, their deployment metadata, the
   result tables, the figures and `task1_predictions.csv` under `models/task1/`.

Every **findings** cell quotes measured numbers from the recorded full run of 9 September
2026 (Kaggle, Tesla T4, Python 3.12.13, torch 2.10.0+cu128, scikit-learn 1.6.1, numpy
2.0.2). Re-running on different hardware moves the third decimal place. It does not move
the ranking.

## 1. Setup and Data Understanding

### 1.1 Thread limits

This cell runs before every other import, and it has to. OpenMP, MKL and OpenBLAS each read
their thread count once, when the shared library is first loaded, and ignore any later
change to the environment. Setting these variables after `import numpy` is a no-op that
looks like it worked, which is why this is a separate cell at the top rather than another
block inside the configuration cell below.

In [ ]:
import os
import platform

# sched_getaffinity is the right question to ask rather than cpu_count: inside a container
# or under taskset it reports the cores this process may actually use. It does not exist on
# Windows or macOS, hence the fallback.
try:
    VISIBLE_CORES = len(os.sched_getaffinity(0))
except AttributeError:
    VISIBLE_CORES = os.cpu_count() or 1

for _variable in ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS",
                  "NUMEXPR_NUM_THREADS", "VECLIB_MAXIMUM_THREADS", "BLIS_NUM_THREADS",
                  "LOKY_MAX_CPU_COUNT"):
    os.environ[_variable] = str(VISIBLE_CORES)

HOST_OS = platform.system()
print(f"Host: {HOST_OS} {platform.machine()} | Python {platform.python_version()}")
print(f"CPU : {VISIBLE_CORES} logical cores, all of them")

### 1.2 Imports and configuration

In [ ]:
import gc
import hashlib
import json
import math
import random
import shutil
import time
import warnings
from itertools import combinations, product
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from PIL import Image, ImageOps

import torch
import torch.nn as nn
import torch.nn.functional as F

import joblib
from skimage.feature import hog
from sklearn.dummy import DummyClassifier
from sklearn.exceptions import ConvergenceWarning
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split
from sklearn.svm import LinearSVC

# Only the warning families that are genuinely noise here are silenced. A blanket
# UserWarning filter would also hide sklearn's "y_pred contains labels not in y_true",
# which is precisely the signal the tail-aware metrics in Section 5 exist to reason about.
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message=".*torch.cuda.amp.*")
warnings.filterwarnings("ignore", message=".*Palette images with Transparency.*")

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)
sns.set_theme(style="whitegrid", context="notebook")

PALETTE = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#8b6fc0", "#e87ba4"]
MUTED = "#6b7280"

In [ ]:
# ---------------------------------------------------------------------------------------
# Every setting reused across the notebook is defined once here, so a change is made in one
# place and the whole protocol reads in one screen.
# ---------------------------------------------------------------------------------------
TARGET = "articleType"
RANDOM_STATE = 42

QUICK_RUN = False        # True: a few rows per class and 1-epoch budgets. Structural check only.
ALLOW_CPU = False        # see Section 1.4; ResNet training on CPU presents as a hang

# --- The deterministic image transform, from Section 3.1 of notebook 00 ----------------
IMAGE_TARGET_SIZE = (60, 80)      # width, height: the catalogue's modal image size
IMAGE_PAD_RGB = (255, 255, 255)   # catalogue background, used when padding

# --- Splits, Section 2.1 ---------------------------------------------------------------
REPORTING_SHARE = 0.20   # held out of everything, scored once in Section 5
TUNING_SHARE = 0.20      # held out of fitting; every search and selection decision uses it

# --- Optimisation, Section 3.4 ---------------------------------------------------------
BATCH_SIZE = 128
SEARCH_EPOCHS = 12       # reduced budget, shared by every arm of both neural grids
CONFIRM_EPOCHS = 40      # full budget, for the winning arm of each neural family
PATIENCE = 8             # early stopping on tuning macro-F1
WARMUP_EPOCHS = 3
LABEL_SMOOTHING = 0.05   # justified by the documented label noise, notebook 00 Section 3.2.2

# --- Augmentation, Section 2.4 ---------------------------------------------------------
AUG_FLIP_PROBABILITY = 0.5
AUG_ROTATION_DEGREES = 10.0
AUG_TRANSLATE_FRACTION = 0.08
AUG_JITTER_STRENGTH = 0.2

# --- Class imbalance, Section 3.4 ------------------------------------------------------
# Two regimes, both trained and both scored, so Section 5 shows the effect of the choice
# instead of asserting it. They are alternatives, not additions: each applies a 1/count
# correction exactly once, and stacking them would correct twice for an effective
# 1/count**2, which starves the head without helping the tail.
#   "reweight"  uniform sampling, inverse-frequency loss weights   (the original recipe)
#   "resample"  class-balanced sampling, unweighted loss
# The difference that matters is not the strength of the correction, which is the same, but
# what the rare rows are seen as: under "reweight" a one-image class is one fixed image
# scaled by 195, under "resample" it is drawn about as often as Tshirts and so arrives
# under a different augmentation draw each time.
IMBALANCE_REGIMES = ("reweight", "resample")
RESAMPLE_POWER = 1.0     # draw weight is 1/count ** this; 0 is uniform, 1 is fully balanced

# --- Per-class decision offsets, Section 4.3 -------------------------------------------
# The SVM decides by argmax over decision_function, so a per-class bias that grows with
# rarity trades head precision for tail recall, which is the trade macro-F1 rewards. One
# scalar is fitted rather than 124 offsets: 14 of the 32 rare classes have no tuning row at
# all, and a free offset per class would be fitted on a single image wherever it could be
# fitted at all. Costs no training, so it is scored as its own arm.
SVM_OFFSET_TAU_GRID = ([round(0.01 * step, 3) for step in range(11)]           # 0.00-0.10
                       + [round(0.10 + 0.05 * step, 3) for step in range(1, 19)])  # 0.15-1.00

# --- Search grids, Section 4.1 ---------------------------------------------------------
NEURAL_GRID = [dict(lr=lr, weight_decay=wd)
               for lr, wd in product((3e-4, 1e-3, 3e-3), (1e-4, 1e-3))]
SVM_GRID = [dict(C=value) for value in (0.003, 0.01, 0.03, 0.1, 0.3, 1.0)]
# Equal budgets are what make the comparison fair: six arms per family, no exceptions.
assert len(NEURAL_GRID) == len(SVM_GRID) == 6

FAMILIES = ("hog_svm", "cnn", "resnet")
NEURAL_FAMILIES = ("cnn", "resnet")

# One arm per thing that gets scored. The neural families appear once per regime; the SVM
# has no sampler to vary, so it appears once and acts as the control for the comparison --
# its row must not move between the regimes, and Section 5 checks that it does not. The
# offset arm reuses the same fitted SVM and differs only at inference.
ARMS = ("hog_svm", "hog_svm_offset") + tuple(
    f"{family}_{regime}" for family in NEURAL_FAMILIES for regime in IMBALANCE_REGIMES)
ARM_FAMILY = {"hog_svm": "hog_svm", "hog_svm_offset": "hog_svm",
              **{f"{f}_{r}": f for f in NEURAL_FAMILIES for r in IMBALANCE_REGIMES}}
ARM_REGIME = {f"{f}_{r}": r for f in NEURAL_FAMILIES for r in IMBALANCE_REGIMES}

# --- Tail-aware reporting buckets, Section 5.2 -----------------------------------------
SUPPORT_BUCKETS = [
    ("head (>=1000)", 1000, np.inf),
    ("body (100-999)", 100, 1000),
    ("tail (10-99)", 10, 100),
    ("rare (<10)", 0, 10),
]

if QUICK_RUN:
    SEARCH_EPOCHS, CONFIRM_EPOCHS, WARMUP_EPOCHS, PATIENCE = 1, 2, 1, 2
    ALLOW_CPU = True     # a structural check is allowed to be slow; it must not be blocked

print(f"QUICK_RUN = {QUICK_RUN} | search {SEARCH_EPOCHS} epochs, confirm {CONFIRM_EPOCHS}")
print(f"Grids: {len(SVM_GRID)} SVM arms and {len(NEURAL_GRID)} neural arms, per family")

### 1.3 Locating the data and the output directory

The manifest is found by walking up from the working directory rather than by naming a
parent. A Jupyter kernel starts in the directory of the notebook it opened, which is
`notebooks/Task1/` here and the repository root under `nbconvert` from the root, so any
rule keyed to one directory name is wrong for the other.

In [ ]:
def find_repository_root():
    """The nearest ancestor directory holding the audited manifest."""
    marker = Path("preprocessed_datasets") / "train_manifest.csv"
    here = Path.cwd().resolve()
    for parent in (here, *here.parents):
        if (parent / marker).is_file():
            return parent
    raise FileNotFoundError(
        f"No {marker} in {here} or any of its parents. Run 00_eda_and_preprocessing.ipynb "
        "first, since its Section 3.4 writes the manifest, and run this notebook from "
        "inside the assignment repository."
    )


REPO_ROOT = find_repository_root()
MANIFEST = REPO_ROOT / "preprocessed_datasets" / "train_manifest.csv"
TRAIN_IMAGE_DIR = REPO_ROOT / "datasets" / "train" / "images_train"
TEST_IMAGE_DIR = REPO_ROOT / "datasets" / "test" / "images_test"
PREDICTION_TEMPLATE = REPO_ROOT / "datasets" / "test" / "styles_prediction.csv"

# One directory holds everything this notebook writes, grouped by what each file is rather
# than by the section that happens to produce it: final/ is what you deploy, tables/ is what
# the report quotes, figures/ is what it prints, predictions/ is what gets submitted.
OUTPUT_DIR = REPO_ROOT / "models" / "task1"
MODEL_DIR = OUTPUT_DIR / "final"
TABLE_DIR = OUTPUT_DIR / "tables"
FIGURE_DIR = OUTPUT_DIR / "figures"
PREDICTION_DIR = OUTPUT_DIR / "predictions"
for _directory in (MODEL_DIR, TABLE_DIR, FIGURE_DIR, PREDICTION_DIR):
    _directory.mkdir(parents=True, exist_ok=True)

for _required in (MANIFEST, TRAIN_IMAGE_DIR, TEST_IMAGE_DIR, PREDICTION_TEMPLATE):
    if not _required.exists():
        raise FileNotFoundError(f"Required input is missing: {_required}")

print("Repository root :", REPO_ROOT)
print("Manifest        :", MANIFEST.relative_to(REPO_ROOT))
print("Writes to       :", OUTPUT_DIR.relative_to(REPO_ROOT))


def save_figure(name):
    """Write the current figure under a registered name.

    Call this immediately BEFORE plt.show(). In the inline backend show() closes the
    figure, so a savefig placed after it writes a blank page: a quiet failure rather than
    an error.
    """
    plt.savefig(FIGURE_DIR / f"{name}.png", dpi=200, bbox_inches="tight")

### 1.4 Device selection

In [ ]:
def choose_device():
    """CUDA where it exists, then Apple Silicon's Metal backend, then CPU.

    MPS is checked through getattr because the attribute is absent on torch builds older
    than 1.12 rather than merely reporting False, and an AttributeError here would be a
    confusing way to learn that.
    """
    if torch.cuda.is_available():
        return torch.device("cuda:0")
    metal = getattr(torch.backends, "mps", None)
    if metal is not None and metal.is_available():
        os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")
        return torch.device("mps")
    return torch.device("cpu")


DEVICE = choose_device()

if DEVICE.type == "cuda":
    _name = torch.cuda.get_device_name(DEVICE)
    _total = torch.cuda.get_device_properties(DEVICE).total_memory / 1e9
    _capability = torch.cuda.get_device_capability(DEVICE)
    print(f"GPU: {_name} | {_total:.1f} GB | compute capability {_capability[0]}.{_capability[1]}")
    print(f"torch {torch.__version__} built against CUDA {torch.version.cuda}")
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True   # autotunes convolutions for this fixed input size
elif DEVICE.type == "cpu" and not ALLOW_CPU:
    raise RuntimeError(
        "No CUDA or MPS device. Training the ResNet on a CPU takes roughly ten minutes per "
        "epoch, which presents as a hang rather than as an error, so the notebook refuses "
        "to start instead. Set ALLOW_CPU = True in Section 1.2 to accept the slowdown "
        "deliberately, or run the HOG + SVM family alone, which never touches the GPU."
    )
else:
    print(f"Device: {DEVICE.type}")

# bf16 has the dynamic range of fp32 and needs no loss scaling. Turing cards such as the T4
# lack it, so fp16 with a gradient scaler is the fallback. Neither changes what is learned.
#
# The capability is read directly rather than through torch.cuda.is_bf16_supported(), which
# grew an `including_emulation=True` default and now answers True on a T4: the card can
# *represent* bf16 by emulating it, at roughly a fifth of the throughput its fp16 tensor
# cores would give, with no error and nothing in the log to say so. Native bf16 starts at
# compute capability 8.0 (Ampere), so that is what gets asked.
AMP_ENABLED = DEVICE.type == "cuda"
NATIVE_BF16 = AMP_ENABLED and torch.cuda.get_device_capability(DEVICE)[0] >= 8
AMP_DTYPE = torch.bfloat16 if NATIVE_BF16 else torch.float16
CHANNELS_LAST = DEVICE.type == "cuda"     # NHWC, the layout the tensor-core kernels want
CACHE_ON_DEVICE = DEVICE.type == "cuda"   # hold the uint8 image cache in VRAM: about 0.6 GB

torch.set_num_threads(VISIBLE_CORES)


def set_seed(seed):
    """Seed every generator this notebook draws from.

    Torch, NumPy and Python are all seeded because the augmentation, the weight
    initialisation and the batch order each draw from a different one. Without all three, a
    "same seed" rerun is not actually a rerun.
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed(RANDOM_STATE)
print(f"Mixed precision: {AMP_ENABLED}"
      f"{'' if not AMP_ENABLED else ' (' + str(AMP_DTYPE).replace('torch.', '') + ')'}"
      f" | channels_last: {CHANNELS_LAST} | image cache on device: {CACHE_ON_DEVICE}")
print("Seeded at", RANDOM_STATE)

### 1.5 The audited manifest

`00_eda_and_preprocessing.ipynb` resolved the duplicate images, collapsed each duplicate
group to a single row, and wrote the surviving rows to `train_manifest.csv`. This notebook
takes that file as given and filters it to the rows carrying an `articleType` label.

Filtering on one target rather than dropping incomplete rows globally is the rule set out
in Section 3.3 of that notebook: a row missing `usage` is still valid for `articleType`,
and a global filter would impose one task's label gaps on another.

In [ ]:
def load_manifest(target, image_dir=None):
    """The audited manifest, filtered to rows labelled for one target, with a path column."""
    image_dir = Path(image_dir or TRAIN_IMAGE_DIR)
    frame = pd.read_csv(MANIFEST)
    if target not in frame.columns:
        raise KeyError(f"{target!r} is not a manifest column. Available: {list(frame.columns)}")
    frame = frame.loc[frame[target].notna()].reset_index(drop=True)
    # str rather than Path: a Path survives PIL but writes a machine-local absolute path if
    # a split is ever exported to CSV.
    frame["path"] = frame["filename"].map(lambda name: str(image_dir / name))
    return frame


catalogue = load_manifest(TARGET)
print(f"{len(catalogue):,} labelled images | {catalogue[TARGET].nunique()} article types")
display(catalogue[["id", TARGET, "gender", "season", "usage"]].head())

**Manifest findings.** The manifest carries **37,846 rows labelled for `articleType`**
across **124 classes**, one row per resolved duplicate group. Every row names an image that
exists on disk, so no row is dropped later for a missing file.

### 1.6 Class distribution

In [ ]:
class_counts = catalogue[TARGET].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
class_counts.head(20).plot.bar(ax=axes[0], color=PALETTE[0])
axes[0].set(title="20 largest classes", ylabel="Images", xlabel="")
axes[0].tick_params(axis="x", labelrotation=75, labelsize=8)

# Log scale, because a linear axis over a 5,000:1 range renders the entire tail as zero.
axes[1].plot(range(len(class_counts)), class_counts.to_numpy(), color=PALETTE[1])
axes[1].set(title="Every class, ranked", xlabel="Class rank", ylabel="Images", yscale="log")
for threshold in (1000, 100, 10):
    axes[1].axhline(threshold, color=MUTED, lw=0.8, ls="--")

plt.tight_layout()
save_figure("fig01_class_distribution")
plt.show()

In [ ]:
# The support buckets are the unit the Section 5 metric breakdown reports in, so the class
# counts behind them are established here rather than asserted there.
bucket_summary = pd.DataFrame([
    {"Bucket": name,
     "Classes": int(((class_counts >= low) & (class_counts < high)).sum()),
     "Images": int(class_counts[(class_counts >= low) & (class_counts < high)].sum())}
    for name, low, high in SUPPORT_BUCKETS
])
bucket_summary["Share of images %"] = (
    bucket_summary["Images"] / len(catalogue) * 100).round(1)
display(bucket_summary)
print(f"Largest class : {class_counts.index[0]} at {class_counts.iloc[0]:,} images")
print(f"Smallest class: {class_counts.iloc[-1]} image(s)")
print(f"Imbalance ratio: {class_counts.iloc[0] / class_counts.iloc[-1]:,.0f} : 1")

**Distribution findings.** The imbalance is the defining property of this task and it
drives most of the design decisions that follow.

- `Tshirts` holds **5,267 images**; the smallest classes hold **1**. That is a ratio of
  **5,267 : 1**.
- Only **8 classes** have 1,000 images or more, but they account for over a third of the
  catalogue. **35 classes** have fewer than 10 images each.
- A macro-averaged F1 therefore weights a 1-image class exactly as heavily as `Tshirts`.
  That is the intended behaviour here — the assignment asks for article type, not for
  article type among the popular ones — but it means accuracy and macro-F1 will disagree
  sharply, and Section 4.2 shows them selecting different hyperparameters.
- Classes with a single row cannot appear in both a training and an evaluation split.
  Section 2.1 sends them wholly to training, which is why the class counts per split differ.

## 2. Data Preparation

### 2.1 The three-way split

Every model in this notebook is compared on the same rows, and the rows used to *choose* a
model are never the rows used to *report* it. That needs three disjoint sets, not two:

| Split | Share | What it is for |
|---|---|---|
| **fit** | 64% | Training rows for every search arm and every confirmation run |
| **tuning** | 16% | Scores every hyperparameter arm, picks each family's recipe, picks the winner |
| **reporting** | 20% | Held out of all of the above. Scored exactly once, in Section 5 |

The split is drawn twice with the same helper: first `reporting` off the whole manifest,
then `tuning` off what remains. Whole `group_id` values stay on one side, so byte-identical
images cannot straddle a boundary. Notebook 00 already collapsed every duplicate group, so
in practice this is a verified invariant rather than an active control — the assertion is
what verifies it.

Classes represented by fewer than two groups go entirely to training, because a class with
one example cannot also be evaluated. Holding them out of the stratified draw is why the
achieved share sits slightly below the requested 20%.

In [ ]:
def make_split(frame, target, validation_share, random_state):
    """Split a task frame into training and validation rows, stratified, grouped.

    Returns:
        (training frame, validation frame), each with a fresh index.
    """
    frame = frame.loc[frame[target].notna()].reset_index(drop=True)
    assert frame["group_id"].is_unique, (
        "Multi-row group_id values found. Section 3.2 of notebook 00 should have collapsed "
        "every duplicate group; re-run it before splitting."
    )

    group_label = frame.groupby("group_id")[target].agg(
        lambda values: values.value_counts().index[0])
    groups_per_class = group_label.value_counts()
    # A class with one group cannot be split, so it goes wholly to training.
    splittable = group_label[~group_label.isin(groups_per_class[groups_per_class < 2].index)]

    _, validation_groups = train_test_split(
        splittable.index, test_size=validation_share,
        stratify=splittable.values, random_state=random_state)

    is_validation = frame["group_id"].isin(set(validation_groups))
    return (frame.loc[~is_validation].reset_index(drop=True),
            frame.loc[is_validation].reset_index(drop=True))


def check_disjoint(*frames):
    """Fail loudly if any id or group_id appears in more than one split."""
    for left, right in combinations(frames, 2):
        for column in ("id", "group_id"):
            overlap = set(left[column].astype(str)) & set(right[column].astype(str))
            assert not overlap, f"Leakage across splits on {column}: {sorted(overlap)[:5]}"

In [ ]:
# Reporting is drawn first, off the whole manifest, and then never touched again until
# Section 5. Tuning is drawn from what is left, with a different seed so the two draws are
# independent rather than nested copies of one ordering.
labelled_frame, report_frame = make_split(
    catalogue, TARGET, REPORTING_SHARE, RANDOM_STATE)
fit_frame, tune_frame = make_split(
    labelled_frame, TARGET, TUNING_SHARE, RANDOM_STATE + 1)

check_disjoint(fit_frame, tune_frame, report_frame)

if QUICK_RUN:
    # Keep every class present in training, even in a structural run.
    fit_frame = fit_frame.groupby(TARGET, group_keys=False).head(5).reset_index(drop=True)
    tune_frame = tune_frame.groupby(TARGET, group_keys=False).head(2).reset_index(drop=True)
    report_frame = report_frame.groupby(TARGET, group_keys=False).head(2).reset_index(drop=True)
    labelled_frame = pd.concat([fit_frame, tune_frame], ignore_index=True)

# The class order is fixed here, once, and every model, table and prediction below uses it.
# Sorting rather than taking pandas' encounter order makes the mapping reproducible.
CLASSES = sorted(labelled_frame[TARGET].unique())
CLASS_TO_INDEX = {label: index for index, label in enumerate(CLASSES)}
N_CLASSES = len(CLASSES)
CLASS_SUPPORT = np.array([(labelled_frame[TARGET] == name).sum() for name in CLASSES])

# A class the models can never predict would silently cost macro-F1 on every split.
assert set(CLASSES) == set(fit_frame[TARGET]), "A class is absent from the fitting rows."

split_summary = pd.DataFrame([
    {"Split": name, "Rows": len(frame),
     "Share %": round(len(frame) / len(catalogue) * 100, 1),
     "Classes present": frame[TARGET].nunique()}
    for name, frame in (("fit", fit_frame), ("tuning", tune_frame), ("reporting", report_frame))
])
display(split_summary)

for _name, _frame in (("fit", fit_frame), ("tuning", tune_frame), ("reporting", report_frame)):
    _frame[["id", TARGET, "group_id"]].to_csv(TABLE_DIR / f"split_{_name}.csv", index=False)
print("Split membership written to", TABLE_DIR.name + "/split_*.csv")

**Split findings.** The draw gives **24,223 fitting rows, 6,055 tuning rows and 7,568
reporting rows**, which is 64.0% / 16.0% / 20.0% of the 37,846 labelled images.

All **124 classes** appear in the fitting rows, as the assertion requires, but only **106**
appear in tuning and **110** in reporting. The missing ones are the rare classes that hold
too few groups to be split. This matters for how the metrics are read: a class absent from
an evaluation split contributes nothing to that split's macro-F1 rather than contributing a
zero, so the macro-F1 figures below are averages over the classes actually present, and the
tuning and reporting figures average over slightly different class sets. They are therefore
comparable *within* a split and across models, which is all the selection rule needs, but
not directly comparable *between* splits.

### 2.2 The image transform and the cache

One deterministic transform, applied exactly once per image: convert to RGB, then pad to
60x80 preserving the portrait aspect ratio. Padding rather than cropping or stretching is
argued in Section 3.1 of notebook 00 — the portrait shape carries class cues at both the
top and the bottom of a product, and 38,595 of 38,612 images are already at the target size
and so pass through unresampled.

Decoding 37,846 JPEGs takes minutes and would otherwise happen once per epoch, so each
split is decoded once into a single `uint8` array. Only the deterministic transform is
frozen by this. Augmentation still happens per batch, per epoch, so nothing about the
training distribution is fixed by caching.

In [ ]:
def standardize_image(image, target_size=IMAGE_TARGET_SIZE):
    """The deterministic input transform. Augmentation comes after, normalisation last."""
    image = ImageOps.exif_transpose(image).convert("RGB")
    if image.size == tuple(target_size):
        return image                      # already the target size: no resampling at all
    return ImageOps.pad(image, tuple(target_size), method=Image.Resampling.BILINEAR,
                        color=IMAGE_PAD_RGB, centering=(0.5, 0.5))


def build_image_cache(frame, description):
    """Decode a frame's images once into one uint8 array, in the frame's row order.

    Returns:
        Array of shape (rows, height, width, 3), dtype uint8.
    """
    width, height = IMAGE_TARGET_SIZE
    images = np.empty((len(frame), height, width, 3), dtype=np.uint8)
    start = time.time()
    for position, path in enumerate(frame["path"]):
        with Image.open(path) as handle:
            images[position] = np.asarray(standardize_image(handle))
        if position and position % 10000 == 0:
            print(f"  {description}: {position:,} / {len(frame):,}")
    print(f"{description}: {len(frame):,} images in {time.time() - start:.0f}s "
          f"({images.nbytes / 1e6:.0f} MB)")
    return images


fit_images = build_image_cache(fit_frame, "fitting")
tune_images = build_image_cache(tune_frame, "tuning")

In [ ]:
def to_indices(frame):
    """Class indices for a frame, in the fixed CLASSES order."""
    mapped = frame[TARGET].map(CLASS_TO_INDEX)
    assert mapped.notna().all(), "A row carries a class absent from CLASSES."
    return mapped.to_numpy(dtype=np.int64)


y_fit, y_tune = to_indices(fit_frame), to_indices(tune_frame)
# The classes actually present in tuning. Scoring against these rather than against all 124
# is what keeps a class that cannot appear from being counted as a zero.
SCOREABLE = np.unique(y_tune)
print(f"y_fit {y_fit.shape} | y_tune {y_tune.shape} | scoreable classes {len(SCOREABLE)}")

### 2.3 Normalisation

Per-channel mean and standard deviation, fitted **on the fitting rows only** and then
applied unchanged to tuning, reporting and test. Fitting them on all the data would leak
the evaluation rows' pixel statistics into training.

Computed in one streaming pass from running sums, so memory does not scale with the number
of images.

In [ ]:
def normalisation_from(images):
    """Per-channel (mean, std) over a uint8 image cache, on the [0, 1] scale."""
    total, squares, n_pixels = np.zeros(3), np.zeros(3), 0
    for begin in range(0, len(images), 1024):
        block = images[begin:begin + 1024].astype(np.float32) / 255.0
        total += block.sum((0, 1, 2), dtype=np.float64)
        squares += np.einsum("nhwc,nhwc->c", block, block, dtype=np.float64)
        n_pixels += int(np.prod(block.shape[:3]))
    mean = total / n_pixels
    # Clamped away from zero: a constant channel would otherwise divide by zero.
    std = np.maximum(np.sqrt(np.maximum(squares / n_pixels - mean ** 2, 0)), 1e-6)
    return mean.astype(np.float32), std.astype(np.float32)


def set_normalisation(mean, std):
    """Publish the constants as both arrays and broadcastable device tensors."""
    global NORM_MEAN, NORM_STD, NORM_MEAN_T, NORM_STD_T
    NORM_MEAN, NORM_STD = np.asarray(mean, np.float32), np.asarray(std, np.float32)
    NORM_MEAN_T = torch.tensor(NORM_MEAN, device=DEVICE).view(1, 3, 1, 1)
    NORM_STD_T = torch.tensor(NORM_STD, device=DEVICE).view(1, 3, 1, 1)


set_normalisation(*normalisation_from(fit_images))
print("Mean (R, G, B):", np.round(NORM_MEAN, 4))
print("Std  (R, G, B):", np.round(NORM_STD, 4))

**Normalisation findings.** The fitting rows give a mean of **(0.8493, 0.8325, 0.8267)**
and a standard deviation of **(0.2716, 0.2831, 0.2870)**.

A mean above 0.83 in every channel is the white catalogue background dominating the frame,
not a bright product — most of a 60x80 crop of a shoe on white is white. The near-equal
channel means say the background is neutral rather than tinted. This is also why the
padding colour in Section 1.2 is white: padding with black would have shifted these
constants and put the padded border at the opposite extreme of the normalised range from
the real background.

### 2.4 Augmentation

A catalogue photograph is centred, upright and evenly lit, so the useful augmentations are
small ones. The policy is a horizontal flip, a rotation up to 10 degrees, a translation up
to 8% of the frame, and independent brightness, saturation and contrast jitter at 20%.

Everything runs on the GPU, on a batch that is already there, with independent parameters
per sample. There is no CPU dataloader and no host-to-device copy in the training loop.

In [ ]:
LUMA = torch.tensor([0.299, 0.587, 0.114], device=DEVICE).view(1, 3, 1, 1)


def augment_batch(x):
    """Apply the Section 2.4 policy to a batch, with independent parameters per sample.

    Args:
        x: float tensor of shape (n, 3, height, width) with values in [0, 1].

    Returns:
        A tensor of the same shape and range.
    """
    n = x.shape[0]
    device = x.device

    # torch.where selects per sample, so the flip is genuinely drawn per image.
    flip = torch.rand(n, device=device) < AUG_FLIP_PROBABILITY
    x = torch.where(flip.view(-1, 1, 1, 1), x.flip(-1), x)

    # Rotation and translation in one affine warp. The height/width factors correct for
    # affine_grid's normalised coordinates, without which the rotation would shear.
    height, width = x.shape[-2], x.shape[-1]
    angle = (torch.rand(n, device=device) * 2 - 1) * math.radians(AUG_ROTATION_DEGREES)
    shift_x = (torch.rand(n, device=device) * 2 - 1) * AUG_TRANSLATE_FRACTION * 2
    shift_y = (torch.rand(n, device=device) * 2 - 1) * AUG_TRANSLATE_FRACTION * 2
    cos, sin = torch.cos(angle), torch.sin(angle)

    theta = torch.zeros(n, 2, 3, device=device)
    theta[:, 0, 0] = cos
    theta[:, 0, 1] = -sin * height / width
    theta[:, 0, 2] = shift_x
    theta[:, 1, 0] = sin * width / height
    theta[:, 1, 1] = cos
    theta[:, 1, 2] = shift_y

    grid = F.affine_grid(theta, list(x.shape), align_corners=False)
    # Offset by -1 so grid_sample's zero padding lands on white once shifted back. Rotating
    # a white-background product into a black border would be a stronger signal than the
    # rotation itself.
    x = F.grid_sample(x - 1.0, grid, mode="bilinear", padding_mode="zeros",
                      align_corners=False) + 1.0

    def factor():
        return 1.0 + (torch.rand(n, 1, 1, 1, device=device) * 2 - 1) * AUG_JITTER_STRENGTH

    x = x * factor()                                       # brightness
    grey = (x * LUMA).sum(dim=1, keepdim=True)
    x = (x - grey) * factor() + grey                       # saturation
    mean = grey.mean(dim=(2, 3), keepdim=True)
    x = (x - mean) * factor() + mean                       # contrast

    return x.clamp_(0.0, 1.0)

In [ ]:
# What the network actually sees. Eight draws of one image, so the spread of the policy is
# visible rather than described.
_sample = torch.from_numpy(fit_images[:1].copy()).to(DEVICE)
_sample = _sample.permute(0, 3, 1, 2).float().div_(255.0).repeat(8, 1, 1, 1)
set_seed(RANDOM_STATE)
_augmented = augment_batch(_sample).cpu().permute(0, 2, 3, 1).numpy()

fig, axes = plt.subplots(1, 9, figsize=(13, 2.2))
axes[0].imshow(fit_images[0])
axes[0].set_title("original", fontsize=8)
for index, ax in enumerate(axes[1:]):
    ax.imshow(_augmented[index])
    ax.set_title(f"draw {index + 1}", fontsize=8)
for ax in axes:
    ax.axis("off")
fig.suptitle(f"Augmentation policy applied to one {fit_frame[TARGET].iloc[0]} image", y=1.06)
plt.tight_layout()
save_figure("fig02_augmentation_preview")
plt.show()

**Augmentation findings.** Every draw is still recognisably the same product in the same
pose, which is the intent. The rotation and shift are small enough that the item never
leaves the frame, and the white fill on the rotated corners matches the catalogue
background rather than introducing a black border the network could learn from.

The colour jitter is the one component worth questioning: colour is genuinely predictive
for a few classes here, and jittering saturation by 20% weakens that. It is kept because
the alternative — a model keyed to catalogue lighting — generalises worse to the test
photographs, and because the effect is bounded at 20%.

## 3. Model Design

Three families, chosen to span the space of reasonable answers rather than to stack the
comparison:

1. **HOG + linear SVM** — a classical pipeline with no learned features. It is the baseline
   the neural models have to beat, and on this data it is a serious competitor rather than a
   formality.
2. **PlainCNN** — a VGG-style stack trained from scratch. The reference architecture.
3. **SmallResNet** — ResNet-18's topology with a stem sized for 60x80 inputs, testing
   whether residual connections and greater depth help at this image size.

All three see identical pixels, identical splits and an identically sized search grid.

### 3.1 HOG + linear SVM

Histogram of Oriented Gradients on the luminance channel, then a linear SVM. HOG describes
local edge orientation, which is what separates a shoe silhouette from a shirt silhouette
at this resolution, and it is invariant to the small brightness shifts the catalogue
contains.

The descriptor is computed one image at a time in float32 into a preallocated output. The
obvious vectorised form promotes the whole uint8 cache to float64, which peaks at several GB
for no speed gain. The work is spread across processes because skimage's HOG is a
single-threaded Python-level loop and this is the longest stretch in the notebook during
which the GPU sits idle; chunks are concatenated in order, so the output is identical to the
serial version row for row.

In [ ]:
HOG_PARAMS = dict(
    orientations=9,
    pixels_per_cell=(8, 8),
    cells_per_block=(2, 2),
    block_norm="L2-Hys",
    feature_vector=True,
)

_HOG_LUMA = np.array([0.299, 0.587, 0.114], dtype=np.float32)


def _hog_chunk(chunk):
    """HOG descriptors for a contiguous block of images, in the block's own order.

    Module-level rather than a closure so joblib's process backend can pickle it. Chunked
    rather than one task per image, because 30,000 tasks would spend more time in dispatch
    than in the descriptor.
    """
    features = None
    for position, image in enumerate(chunk):
        grey = (image.astype(np.float32) @ _HOG_LUMA) / 255.0
        descriptor = hog(grey, **HOG_PARAMS).astype(np.float32)
        if features is None:
            features = np.empty((len(chunk), descriptor.size), dtype=np.float32)
        features[position] = descriptor
    return features


def hog_features(images, description, chunk_size=2048):
    """HOG descriptor per image, computed on the luminance channel.

    Returns:
        float32 array of shape (rows, n_features).
    """
    start = time.time()
    bounds = list(range(0, len(images), chunk_size))
    parallel = VISIBLE_CORES > 1 and len(bounds) > 1

    if parallel:
        # A failure here is a throughput problem, not a correctness one, so it falls back
        # rather than taking the cell down with it.
        try:
            blocks = joblib.Parallel(n_jobs=VISIBLE_CORES, prefer="processes")(
                joblib.delayed(_hog_chunk)(images[begin:begin + chunk_size])
                for begin in bounds)
        except Exception as error:                 # noqa: BLE001 - any failure falls back
            print(f"  parallel HOG unavailable, computing serially: {error}")
            parallel = False
    if not parallel:
        blocks = [_hog_chunk(images[begin:begin + chunk_size]) for begin in bounds]

    features = np.concatenate(blocks, axis=0)
    print(f"{description}: {features.shape[0]:,} x {features.shape[1]} features in "
          f"{time.time() - start:.0f}s ({features.nbytes / 1e6:.0f} MB, "
          f"{'parallel' if parallel else 'serial'})")
    return features


def svm_fit(x, y, config):
    """A linear SVM on HOG features, with balanced class weights.

    ConvergenceWarning is promoted to an error. A non-converged SVM that quietly scores a
    little worse would enter model selection as a weaker competitor, and the comparison
    would be reporting an optimisation failure as a property of the model family.
    """
    model = LinearSVC(**config, dual=False, class_weight="balanced",
                      max_iter=20000, random_state=RANDOM_STATE)
    with warnings.catch_warnings():
        warnings.simplefilter("error", ConvergenceWarning)
        model.fit(x, y)
    assert np.array_equal(model.classes_, np.arange(N_CLASSES))
    return model

### 3.2 PlainCNN

Four VGG-style blocks, each two 3x3 convolutions with batch normalisation, the first three
followed by a max-pool. Global average pooling replaces the flattened dense head, which is
what keeps the parameter count at roughly a tenth of a comparable VGG while leaving the
receptive field unchanged.

Batch normalisation is not optional here: without it the deeper blocks do not train at the
learning rates the grid searches.

In [ ]:
class PlainCNN(nn.Module):
    """VGG-style stack sized for 60x80 inputs. The reference architecture."""

    def __init__(self, n_classes=None, width=32, dropout=0.3):
        super().__init__()
        n_classes = n_classes or N_CLASSES

        def block(in_channels, out_channels, pool=True):
            layers = [
                nn.Conv2d(in_channels, out_channels, 3, padding=1, bias=False),
                nn.BatchNorm2d(out_channels), nn.ReLU(inplace=True),
                nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False),
                nn.BatchNorm2d(out_channels), nn.ReLU(inplace=True),
            ]
            if pool:
                layers.append(nn.MaxPool2d(2))
            return nn.Sequential(*layers)

        self.features = nn.Sequential(
            block(3, width),                 # 80x60 -> 40x30
            block(width, width * 2),         # 40x30 -> 20x15
            block(width * 2, width * 4),     # 20x15 -> 10x7
            block(width * 4, width * 8, pool=False),
        )
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(width * 8, n_classes)

    def forward(self, x):
        x = self.pool(self.features(x)).flatten(1)
        return self.fc(self.dropout(x))

### 3.3 SmallResNet

ResNet-18's block structure, with one change: the stem is a stride-1 3x3 convolution rather
than the stride-2 7x7 convolution and max-pool of the original. A 60x80 input passed through
the standard stem arrives at the first residual stage at 15x20, which discards most of the
spatial detail the small classes depend on.

In [ ]:
class BasicBlock(nn.Module):
    """Standard two-convolution residual block with an optional projection shortcut."""

    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, stride, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, 1, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        # A shortcut can only be added to the block output if both match in shape, so a
        # change of stride or width needs a 1x1 projection to bring it into line.
        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 1, stride, bias=False),
                nn.BatchNorm2d(out_channels),
            )

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)), inplace=True)
        out = self.bn2(self.conv2(out))
        return F.relu(out + self.shortcut(x), inplace=True)


class SmallResNet(nn.Module):
    """ResNet-18 topology with a stride-1 3x3 stem, sized for 60x80 inputs."""

    def __init__(self, n_classes=None, width=64, blocks=(2, 2, 2, 2), dropout=0.3):
        super().__init__()
        n_classes = n_classes or N_CLASSES
        self.stem = nn.Sequential(
            nn.Conv2d(3, width, 3, 1, 1, bias=False),
            nn.BatchNorm2d(width), nn.ReLU(inplace=True),
        )
        stages, in_channels = [], width
        for stage_index, n_blocks in enumerate(blocks):
            out_channels = width * (2 ** stage_index)
            for block_index in range(n_blocks):
                # Downsample once per stage, at its first block, and never in stage 0.
                stride = 2 if (block_index == 0 and stage_index > 0) else 1
                stages.append(BasicBlock(in_channels, out_channels, stride))
                in_channels = out_channels
        self.stages = nn.Sequential(*stages)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(in_channels, n_classes)

    def forward(self, x):
        x = self.pool(self.stages(self.stem(x))).flatten(1)
        return self.fc(self.dropout(x))


FACTORIES = {"cnn": PlainCNN, "resnet": SmallResNet}

for _family, _factory in FACTORIES.items():
    _model = _factory()
    print(f"{_family:>7}: {sum(p.numel() for p in _model.parameters()):>10,} parameters")
    del _model

**Architecture findings.** PlainCNN carries **1,205,084 parameters** and SmallResNet
**11,232,444**, so the ResNet is **9.3x the model** at the same input size. Section 4 shows that
the extra capacity does not pay for itself here: with 24,223 training images across 124
classes, the larger network overfits sooner and scores lower on every metric. Depth is not
the binding constraint on this task; data in the tail is.

### 3.4 Shared training policy

Every neural run in this notebook uses the same optimiser, schedule, loss and stopping rule.
Only the learning rate and weight decay vary, and those are exactly what Section 4 searches.
Holding the rest fixed is what makes the two families comparable: a difference in the
results is then a difference between the architectures, not between two training recipes.

- **Optimiser.** AdamW. Decoupled weight decay is the point — with L2 folded into the
  gradient, the decay a parameter receives depends on its Adam moment, so the grid's
  `weight_decay` axis would not mean the same thing at two learning rates.
- **Schedule.** Linear warmup over 3 epochs, then cosine decay to zero. Warmup matters here
  because batch normalisation statistics are meaningless for the first few hundred steps and
  a full-rate update against them destabilises training.
- **Loss.** Class-balanced cross-entropy with 0.05 label smoothing. Weighting by `n / (K *
  count)` is what stops a model from reaching 40% accuracy by answering `Tshirts` and
  ignoring the tail. Smoothing is justified by the label noise documented in notebook 00.
- **Stopping.** Early stopping on tuning macro-F1 with patience 8, restoring the best epoch's
  weights. Macro-F1 rather than accuracy, because accuracy is the metric the imbalance can be
  gamed on.

In [ ]:
class BatchStream:
    """Device-resident batches, replacing Dataset plus DataLoader.

    The images are held once as uint8 in the device's memory and converted to float per
    batch, so nothing is copied from the host during training and the host holds no
    per-batch buffers. At this image size that removes the dataloader from the critical
    path entirely.

    Args:
        images: uint8 array of shape (rows, height, width, 3), or an existing device
            tensor to share with another stream.
        labels: integer class indices aligned to `images`.
        batch_size: rows per batch.
        augment: apply the Section 2.4 policy. Training only.
        shuffle: reorder each epoch.
        sampler: "instance" draws every row once per epoch, "balanced" draws rows with
            probability 1/count so the classes arrive in roughly equal numbers.
    """

    def __init__(self, images, labels, batch_size=BATCH_SIZE, augment=False, shuffle=False,
                 sampler="instance"):
        if torch.is_tensor(images):
            self.images = images                    # shared with another stream, not copied
        else:
            self.images = torch.from_numpy(np.ascontiguousarray(images))
            if CACHE_ON_DEVICE:
                self.images = self.images.to(DEVICE, non_blocking=True)
        self.labels = torch.as_tensor(np.asarray(labels), dtype=torch.long, device=DEVICE)
        self.batch_size = batch_size
        self.augment = augment
        self.shuffle = shuffle

        assert sampler in ("instance", "balanced"), sampler
        self.sampler = sampler
        if sampler == "balanced":
            counts = torch.bincount(self.labels, minlength=N_CLASSES).float()
            # Weight per row, not per class: multinomial draws over rows, so every row of a
            # rare class has to carry that class's full share.
            self.sample_weights = (1.0 / counts.clamp(min=1)) ** RESAMPLE_POWER
            self.sample_weights = self.sample_weights[self.labels]
        else:
            self.sample_weights = None

    def __len__(self):
        return math.ceil(len(self.labels) / self.batch_size)

    def __iter__(self):
        if self.sampler == "balanced":
            # With replacement, and the epoch keeps its length: this changes which rows the
            # arm sees, not how many gradient steps it gets, so the budget stays equal to
            # the other regime's. A one-image class has to be drawn many times over for the
            # balance to hold at all, and each draw is augmented independently.
            order = torch.multinomial(self.sample_weights, len(self.labels),
                                      replacement=True).to(self.images.device)
        elif self.shuffle:
            order = torch.randperm(len(self.labels), device=self.images.device)
        else:
            order = torch.arange(len(self.labels), device=self.images.device)
        for start in range(0, len(order), self.batch_size):
            index = order[start:start + self.batch_size]
            batch = self.images[index].to(DEVICE, non_blocking=True)
            x = batch.permute(0, 3, 1, 2).float().div_(255.0)
            if self.augment:
                x = augment_batch(x)
            # Normalisation is applied last, after augmentation, so the jitter operates on
            # the [0, 1] scale the policy was designed against.
            x = (x - NORM_MEAN_T) / NORM_STD_T
            if CHANNELS_LAST:
                x = x.contiguous(memory_format=torch.channels_last)
            yield x, self.labels[index.to(self.labels.device)]

In [ ]:
def make_criterion(y, regime):
    """The loss for one imbalance regime.

    Under "reweight", weighting each class by n / (K * count) makes the 1-image classes
    contribute as much total loss as Tshirts does. Without it a model reaches respectable
    accuracy by learning the eight head classes and answering them for everything, which is
    exactly the failure macro-F1 is chosen to expose.

    Under "resample" the same criterion is returned unweighted, because that regime's
    BatchStream has already equalised the classes by drawing them equally often. Weighting
    on top of balanced sampling would apply 1/count twice.
    """
    assert regime in IMBALANCE_REGIMES, regime
    if regime == "resample":
        def criterion(logits, targets):
            return F.cross_entropy(logits, targets, label_smoothing=LABEL_SMOOTHING)
        return criterion

    counts = np.bincount(y, minlength=N_CLASSES)
    assert (counts > 0).all(), "A class has no fitting rows; the weight would be infinite."
    weights = torch.as_tensor(len(y) / (N_CLASSES * counts), device=DEVICE, dtype=torch.float32)

    def criterion(logits, targets):
        losses = F.cross_entropy(logits, targets, reduction="none",
                                 label_smoothing=LABEL_SMOOTHING)
        # Mean over rows rather than a weighted mean, so the loss scale does not drift with
        # whichever classes a batch happened to draw.
        return (losses * weights[targets]).mean()

    return criterion


def balanced_criterion(y):
    """The inverse-frequency criterion under the name the original recipe used."""
    return make_criterion(y, "reweight")


def score_predictions(y_true, y_pred):
    """Macro-F1, accuracy and weighted F1 over the classes actually present in y_true."""
    assert len(y_true) == len(y_pred) and len(y_true)
    present = np.unique(y_true)
    return dict(
        macro_f1=float(f1_score(y_true, y_pred, labels=present, average="macro", zero_division=0)),
        accuracy=float(accuracy_score(y_true, y_pred)),
        weighted_f1=float(f1_score(y_true, y_pred, labels=present, average="weighted", zero_division=0)),
    )


def choose_best(rows):
    """The winning row under the selection rule, declared once and applied everywhere.

    Highest macro-F1; ties broken by accuracy, then by the candidate identifier. The final
    tie-break is lexical rather than arbitrary so that the rule is deterministic: two arms
    that score identically must not select differently on a rerun.
    """
    assert rows and all(np.isfinite(row["macro_f1"]) for row in rows)
    return sorted(rows, key=lambda row: (-row["macro_f1"], -row["accuracy"], row["candidate"]))[0]


def prepare_model(model):
    """Move a model to the device in the memory layout the convolution kernels prefer."""
    model = model.to(DEVICE)
    if CHANNELS_LAST:
        model = model.to(memory_format=torch.channels_last)
    return model


def make_scaler():
    """A gradient scaler, enabled only for fp16. bf16 has fp32's range and needs no scaling."""
    enabled = AMP_ENABLED and AMP_DTYPE == torch.float16
    try:
        return torch.amp.GradScaler(DEVICE.type, enabled=enabled)
    except (AttributeError, TypeError):     # torch < 2.4 keeps it under torch.cuda.amp
        return torch.cuda.amp.GradScaler(enabled=enabled)


def release(*objects):
    """Drop references the notebook no longer reads and return their VRAM."""
    del objects
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

In [ ]:
def run_epoch(model, stream, criterion, optimiser=None, scaler=None):
    """One pass over a stream. Trains if an optimiser is given, otherwise evaluates.

    Returns:
        (mean loss, logits array or None). Logits come back in float32 whatever the
        autocast dtype was, so every metric downstream sees the same precision.

    The running loss stays on the device until the pass ends. Calling .item() per batch
    forces a host synchronisation on every step, and at this model size the step is short
    enough for that stall to be a real share of the epoch. One synchronisation per epoch
    reports the identical number.
    """
    training = optimiser is not None
    model.train(training)
    loss_sum = torch.zeros((), device=DEVICE, dtype=torch.float32)
    n_seen, collected = 0, []

    with torch.set_grad_enabled(training):
        for images, targets in stream:
            with torch.autocast(device_type=DEVICE.type, dtype=AMP_DTYPE, enabled=AMP_ENABLED):
                logits = model(images)
                loss = criterion(logits, targets)

            if training:
                optimiser.zero_grad(set_to_none=True)
                if scaler is not None and scaler.is_enabled():
                    scaler.scale(loss).backward()
                    scaler.step(optimiser)
                    scaler.update()
                else:
                    loss.backward()
                    optimiser.step()
            else:
                collected.append(logits.detach().float())

            loss_sum += loss.detach().float() * targets.shape[0]
            n_seen += targets.shape[0]

    mean_loss = (loss_sum / max(n_seen, 1)).item()
    return mean_loss, (torch.cat(collected).cpu().numpy() if collected else None)


def train(model, train_stream, val_stream, y_val, epochs, lr, weight_decay,
          label, patience=PATIENCE, scoreable=None, regime="reweight"):
    """Train with warmup and cosine decay, early stopping on validation macro-F1.

    The best epoch's weights are restored before returning, so the model handed back is the
    model whose score is reported rather than whatever the last epoch happened to leave.

    `regime` selects the imbalance handling, and it has to agree with the sampler the
    training stream was built with: the criterion is weighted only when the stream is not
    balanced. Passing a balanced stream with regime="reweight" would correct twice.

    Returns:
        (history DataFrame, best validation logits, best macro-F1, best epoch).
    """
    scoreable = SCOREABLE if scoreable is None else scoreable
    # The weights come from the rows this call actually trains on, so the same function
    # serves the search (fitting rows) and the Section 5 refit (fitting + tuning rows).
    criterion = make_criterion(train_stream.labels.cpu().numpy(), regime)
    optimiser = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scaler = make_scaler()

    def schedule(epoch):
        if epoch < WARMUP_EPOCHS:
            return (epoch + 1) / max(WARMUP_EPOCHS, 1)
        progress = (epoch - WARMUP_EPOCHS) / max(epochs - WARMUP_EPOCHS, 1)
        return 0.5 * (1 + np.cos(np.pi * progress))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimiser, schedule)
    history, best = [], {"macro_f1": -1.0, "epoch": -1, "state": None, "logits": None}
    start = time.time()

    for epoch in range(epochs):
        epoch_start = time.time()
        # Read before the scheduler advances. param_groups holds the NEXT epoch's rate once
        # scheduler.step() has run, so reading it after the step records every epoch against
        # the rate the following epoch would use.
        epoch_lr = optimiser.param_groups[0]["lr"]
        train_loss, _ = run_epoch(model, train_stream, criterion, optimiser, scaler)
        val_loss, val_logits = run_epoch(model, val_stream, criterion)
        scheduler.step()

        val_pred = val_logits.argmax(axis=1)
        macro = f1_score(y_val, val_pred, labels=scoreable, average="macro", zero_division=0)
        history.append({"epoch": epoch + 1, "train loss": train_loss, "val loss": val_loss,
                        "val accuracy": accuracy_score(y_val, val_pred),
                        "val macro-F1": macro, "lr": epoch_lr,
                        "seconds": time.time() - epoch_start})

        if macro > best["macro_f1"]:
            best.update({"macro_f1": macro, "epoch": epoch + 1, "logits": val_logits,
                         "state": {k: v.detach().cpu().clone()
                                   for k, v in model.state_dict().items()}})

        print(f"  [{label}] epoch {epoch + 1:>3}/{epochs}  train {train_loss:.3f}  "
              f"val {val_loss:.3f}  acc {history[-1]['val accuracy']:.3f}  "
              f"macro-F1 {macro:.4f}  {history[-1]['seconds']:.0f}s")

        if epoch + 1 - best["epoch"] >= patience:
            print(f"  [{label}] early stop at epoch {epoch + 1}; best was epoch {best['epoch']}")
            break

    model.load_state_dict(best["state"])          # report and save the same weights
    print(f"  [{label}] best macro-F1 {best['macro_f1']:.4f} at epoch {best['epoch']} "
          f"({time.time() - start:.0f}s total)")
    return pd.DataFrame(history), best["logits"], best["macro_f1"], best["epoch"]

## 4. Hyperparameter Tuning and Model Selection

### 4.1 The search protocol

Each family gets **six arms**, scored on the tuning split, and the arms are equal in count
by construction — the assertion in Section 1.2 is what enforces it. An unequal budget would
mean the winner was partly a function of who got more attempts.

| Family | Axis searched | Values |
|---|---|---|
| `hog_svm` | SVM regularisation `C` | 0.003, 0.01, 0.03, 0.1, 0.3, 1.0 |
| `cnn`, `resnet` | learning rate x weight decay | {3e-4, 1e-3, 3e-3} x {1e-4, 1e-3} |

The neural search runs at a **reduced 12-epoch budget**, because six arms at the full
40 epochs would be six times the cost for a ranking that the short runs already separate
clearly. The winning arm of each neural family is then **confirmed at the full 40 epochs**,
and it is that confirmation score, not the search score, that enters model selection. The
SVM has no epoch budget, so its search score is directly comparable and it enters selection
as it stands.

Nothing in this section ever reads a reporting row.

In [ ]:
# Both neural families train against the same streams, built once here rather than per arm:
# the uint8 cache is already on the device and rebuilding it per arm would dominate the
# search. The training stream augments and shuffles; the validation stream does neither.
fit_stream = BatchStream(fit_images, y_fit, BATCH_SIZE, augment=True, shuffle=True)

# The second regime shares the first stream's device cache rather than uploading a second
# copy of the same 24,223 images; only the draw order differs between them.
fit_streams = {
    "reweight": fit_stream,
    "resample": BatchStream(fit_stream.images, y_fit, BATCH_SIZE, augment=True,
                            shuffle=True, sampler="balanced"),
}
tune_stream = BatchStream(tune_images, y_tune, 512)
print(f"fit stream: {len(fit_stream)} batches of {BATCH_SIZE} | "
      f"tuning stream: {len(tune_stream)} batches of 512")

# What the balanced sampler actually does to an epoch, stated in rows rather than trusted.
_drawn = np.bincount(y_fit[torch.multinomial(
    fit_streams["resample"].sample_weights, len(y_fit), replacement=True).cpu().numpy()],
    minlength=N_CLASSES)
_counts = np.bincount(y_fit, minlength=N_CLASSES)
print(f"balanced draw, rarest class: {_counts.min()} row(s) in the split -> "
      f"~{_drawn[_counts.argmin()]} draws per epoch "
      f"(uniform sampling would give {_counts.min()})")

### 4.2 HOG + SVM search

In [ ]:
# The descriptor is computed once per split and reused across all six arms. Recomputing it
# per arm would spend about twenty minutes producing identical features six times.
hog_fit = hog_features(fit_images, "fit HOG")
hog_tune = hog_features(tune_images, "tuning HOG")

In [ ]:
svm_rows = []
svm_tune_scores = {}
for index, config in enumerate(SVM_GRID):
    start = time.time()
    model = svm_fit(hog_fit, y_fit, config)
    # LinearSVC decides by argmax over these one-vs-rest margins, so keeping the scores
    # costs one pass instead of two and gives Section 4.6 the surface it shifts. The row
    # below is the identical number model.predict would have produced.
    scores = model.decision_function(hog_tune)
    svm_tune_scores[f"hog_svm_{index}"] = scores
    row = dict(candidate=f"hog_svm_{index}", family="hog_svm", regime=None, config=config,
               epochs=None, **score_predictions(y_tune, scores.argmax(1)))
    svm_rows.append(row)
    print(f"  C={config['C']:<6} macro-F1 {row['macro_f1']:.4f}  "
          f"accuracy {row['accuracy']:.4f}  ({time.time() - start:.0f}s)")
    release(model)

hog_contender = choose_best(svm_rows)
hog_contender["arm"] = "hog_svm"
display(pd.DataFrame(svm_rows).drop(columns="epochs"))
print("Selected HOG/SVM arm:", hog_contender["candidate"], hog_contender["config"])

**HOG/SVM search findings.** `C = 0.03` wins on macro-F1 at **0.6723**, and this is the one
place in the notebook where the choice of metric visibly changes the answer.

| C | macro-F1 | accuracy |
|---|---|---|
| 0.003 | 0.5976 | 0.7617 |
| 0.01 | 0.6329 | 0.7886 |
| **0.03** | **0.6723** | 0.8015 |
| 0.1 | 0.6634 | 0.8088 |
| 0.3 | 0.6490 | **0.8089** |
| 1.0 | 0.6106 | 0.7993 |

Accuracy peaks at `C = 0.3` and macro-F1 at `C = 0.03`, a full decade apart. Stronger
regularisation keeps the balanced class weights influential and protects the tail; weaker
regularisation lets the head classes dominate the margin, which buys accuracy at the tail's
expense. Selecting on accuracy would have chosen a model **0.023 macro-F1 worse** on exactly
the classes the macro average exists to protect. The rule was fixed before the search ran,
which is what makes that a finding rather than a choice made after seeing the table.

### 4.3 Per-class decision offsets for the SVM

The SVM decides by argmax over its one-vs-rest margins. Nothing forces those margins onto a
common scale across classes, and a class with forty training images sits at a systematic
disadvantage to one with four thousand however the class weights were set during fitting.

A per-class bias that grows with rarity corrects that at inference, for no training cost at
all: `score_c - tau * log(prior_c)`. One scalar is fitted rather than 124 free offsets,
because 14 of the 32 rare classes have no tuning row to fit against at all and most of the
rest have exactly one. A free offset per class would be fitted on a single image wherever it
could be fitted, which is not fitting.

`tau` is chosen here, on the tuning split and against the search-stage model, then frozen.
Section 5 replays it against the refitted SVM exactly as it replays a selected epoch count,
and the shifted arm is carried into selection as a candidate in its own right rather than
being assumed to be an improvement.

In [ ]:
# Fitted against the search-stage SVM, the one trained on the fitting rows only, so the
# tuning split is still held out of everything the offset has seen. Fitting tau against the
# model refitted in Section 5 would be fitting on that model's own training rows, and the
# offset would read better here than it could ever perform on the reporting split.
fit_log_prior = np.log(np.bincount(y_fit, minlength=N_CLASSES) / len(y_fit))
base_scores = svm_tune_scores[hog_contender["candidate"]]

offset_rows = []
for tau in SVM_OFFSET_TAU_GRID:
    shifted = base_scores - tau * fit_log_prior
    offset_rows.append(dict(tau=tau, **score_predictions(y_tune, shifted.argmax(1))))

offset_table = pd.DataFrame(offset_rows)
best_offset = offset_table.loc[offset_table["macro_f1"].idxmax()]
SVM_OFFSET_TAU = float(best_offset["tau"])
offset_table.to_csv(TABLE_DIR / "svm_offset_tau.csv", index=False)

# The offset arm is the same fitted SVM read differently, so it inherits the contender row
# and overrides only what the shift changes.
hog_offset_contender = dict(hog_contender)
hog_offset_contender.update(candidate="hog_svm_offset", arm="hog_svm_offset",
                            macro_f1=float(best_offset["macro_f1"]),
                            accuracy=float(best_offset["accuracy"]),
                            weighted_f1=float(best_offset["weighted_f1"]))

fig, ax = plt.subplots(figsize=(6.5, 3.4))
ax.plot(offset_table["tau"], offset_table["macro_f1"], color=PALETTE[0], label="macro-F1")
ax.plot(offset_table["tau"], offset_table["accuracy"], color=PALETTE[1], ls="--",
        label="accuracy")
ax.axvline(SVM_OFFSET_TAU, color=MUTED, ls=":", label=f"selected tau = {SVM_OFFSET_TAU}")
ax.set(title="Per-class decision offset, tuning split", xlabel="tau", ylabel="Score")
ax.legend(fontsize=8)
plt.tight_layout()
save_figure("fig06_svm_offset_tau")
plt.show()

print(f"tau = {SVM_OFFSET_TAU}: tuning macro-F1 {hog_contender['macro_f1']:.4f} -> "
      f"{best_offset['macro_f1']:.4f} ({best_offset['macro_f1'] - hog_contender['macro_f1']:+.4f}), "
      f"accuracy {hog_contender['accuracy']:.4f} -> {best_offset['accuracy']:.4f}")

### 4.4 CNN and ResNet search

In [ ]:
def neural_arm(family, config, epochs, label, patience=None, regime="reweight"):
    """Train one arm from scratch and score it on the tuning split.

    Every arm is reseeded identically before it builds its model, so the arms differ by
    their hyperparameters and by nothing else — not by where the global RNG happened to be
    when the previous arm finished.
    """
    set_seed(RANDOM_STATE)
    model = prepare_model(FACTORIES[family]())
    history, logits, macro, best_epoch = train(
        model, fit_streams[regime], tune_stream, y_tune, epochs=epochs,
        patience=epochs if patience is None else patience, label=label,
        regime=regime, **config)
    history.to_csv(TABLE_DIR / f"history_{label}.csv", index=False)
    row = dict(candidate=label, family=family, regime=regime, config=config,
               epochs=best_epoch, **score_predictions(y_tune, logits.argmax(1)))
    release(model)
    return row, history

In [ ]:
search_rows = list(svm_rows)
neural_search = {}

for family in ("cnn", "resnet"):
    print(f"\n=== {family}: {len(NEURAL_GRID)} arms at {SEARCH_EPOCHS} epochs ===")
    rows = []
    for index, config in enumerate(NEURAL_GRID):
        # patience = epochs: no early stopping inside the search, so every arm gets the
        # identical budget and a slow starter is not eliminated for starting slowly.
        row, _ = neural_arm(family, config, SEARCH_EPOCHS, f"{family}_search_{index}")
        rows.append(row)
    neural_search[family] = rows
    search_rows.extend(rows)
    display(pd.DataFrame(rows))

**Neural search findings.** Both families select the **lowest learning rate in the grid,
3e-4**, and both collapse at the highest.

| lr | wd | CNN macro-F1 | ResNet macro-F1 |
|---|---|---|---|
| 3e-4 | 1e-4 | 0.2840 | **0.2567** |
| 3e-4 | 1e-3 | **0.2870** | 0.2330 |
| 1e-3 | 1e-4 | 0.2719 | 0.1171 |
| 1e-3 | 1e-3 | 0.2554 | 0.0819 |
| 3e-3 | 1e-4 | 0.0446 | 0.0461 |
| 3e-3 | 1e-3 | 0.0721 | 0.0031 |

At 3e-3 the ResNet reaches **0.0031** macro-F1, which is a diverged run rather than a poorly
tuned one. The ResNet degrades faster than the CNN at every step up the learning-rate axis,
which is the expected signature of the deeper network with 9.3x the parameters on a dataset
this size.

Weight decay is close to irrelevant by comparison: it moves macro-F1 by at most 0.03 at a
fixed learning rate, against the 0.28 that the learning rate moves it. If this grid were
re-cut, the budget should go to extending the learning rate downward — 3e-4 sits at the edge
of the grid and the search gives no evidence that it is a maximum rather than the best of
what was offered. That is a real limitation of the search, and Section 7 restates it.

These absolute numbers are low because the arms only ran 12 epochs. The confirmation runs
below take the winning arms to the full budget, and roughly double them.

### 4.5 Confirmation at the full budget

In [ ]:
contenders = [hog_contender, hog_offset_contender]
confirm_histories = {}

for family in NEURAL_FAMILIES:
    selected = choose_best(neural_search[family])
    # The learning rate and weight decay were searched under "reweight" and are reused for
    # "resample" rather than searched again. That is the honest description of the run: a
    # second six-arm search per family would cost about ninety minutes, and holding the
    # recipe fixed is also what isolates the regime as the one thing that differs.
    for regime in IMBALANCE_REGIMES:
        arm = f"{family}_{regime}"
        print(f"\n=== {arm} confirmation: {selected['config']} at {CONFIRM_EPOCHS} epochs ===")
        # Early stopping is enabled here, unlike in the search: the point of the
        # confirmation run is the best achievable score for this recipe, and the epoch it
        # was reached at is what the Section 5 refit replays.
        row, history = neural_arm(family, selected["config"], CONFIRM_EPOCHS,
                                  f"{arm}_confirm", patience=PATIENCE, regime=regime)
        row["arm"] = arm
        contenders.append(row)
        confirm_histories[arm] = history

candidate_table = pd.DataFrame(contenders)
display(candidate_table)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
# Solid for the original regime, dashed for balanced sampling, one colour per family, so
# the pairing that carries the comparison is the one the eye picks up first.
for (arm, history), colour in zip(confirm_histories.items(), PALETTE):
    style = "-" if arm.endswith("reweight") else "--"
    axes[0].plot(history["epoch"], history["train loss"], color=colour, ls=style, label=arm)
    axes[1].plot(history["epoch"], history["val macro-F1"], color=colour, ls=style, label=arm)
axes[0].set(title="Training loss", xlabel="Epoch", ylabel="Cross-entropy")
axes[1].set(title="Tuning macro-F1", xlabel="Epoch", ylabel="Macro-F1")
axes[1].axhline(hog_contender["macro_f1"], color=MUTED, ls=":", label="HOG + SVM")
for ax in axes:
    ax.legend(fontsize=8)
fig.suptitle("Confirmation runs at the full 40-epoch budget", y=1.03)
plt.tight_layout()
save_figure("fig03_confirmation_curves")
plt.show()

> **Stale — regenerate from this run's tables.** The figures below are from the
> single-arm run committed at `e3322f58`, before the imbalance arms were added.

**Confirmation findings.** The full budget roughly doubles both neural families, and neither
reaches the classical baseline.

| Candidate | Config | Best epoch | macro-F1 | accuracy |
|---|---|---|---|---|
| `hog_svm_2` | C = 0.03 | — | **0.6723** | 0.8015 |
| `cnn_confirm` | lr 3e-4, wd 1e-3 | 39 of 40 | 0.5262 | 0.6352 |
| `resnet_confirm` | lr 3e-4, wd 1e-4 | 34 of 40 | 0.4873 | 0.5699 |

The CNN's best epoch is **39 of 40** — it was still improving when the budget ran out, so its
0.5262 is a floor rather than a converged value. The ResNet peaked at **34** and then drifted,
which together with its wider train/validation loss gap is the overfitting the parameter
count predicted.

Even granting the CNN a longer run, the gap to close is **0.146 macro-F1**, and the curve is
flattening. The honest reading is that 24,223 images across 124 classes is not enough data to
train these networks from scratch to the point where they beat a hand-designed edge
descriptor. HOG encodes the silhouette prior directly; the networks have to learn it, and at
this scale they only partly do.

### 4.6 Selection

In [ ]:
# One rule, applied to every confirmed arm. Everything after this point is reporting: the
# decision is frozen here, before any reporting row has been touched. The imbalance regime
# competes on the same footing as any other choice rather than being assumed to help.
winner = choose_best(contenders)

selection = dict(
    rule="highest tuning macro-F1; ties broken by accuracy, then candidate identifier",
    winner=winner["arm"],
    winner_family=winner["family"],
    contenders=contenders,
    recipes={row["arm"]: dict(config=row["config"], epochs=row["epochs"],
                              regime=row.get("regime")) for row in contenders},
    svm_offset_tau=SVM_OFFSET_TAU,
    tuning_normalisation=[NORM_MEAN.tolist(), NORM_STD.tolist()],
)

assert set(selection["recipes"]) == set(ARMS)
(OUTPUT_DIR / "selection.json").write_text(json.dumps(selection, indent=2), encoding="utf-8")
pd.DataFrame(search_rows).to_csv(TABLE_DIR / "search_all_models.csv", index=False)
candidate_table.to_csv(TABLE_DIR / "selection_candidates.csv", index=False)

print("Selected arm:", selection["winner"])
for arm, recipe in selection["recipes"].items():
    print(f"  {arm:>16}: {recipe}")

In [ ]:
# The tuning caches and streams are finished with. Releasing them here returns roughly a
# gigabyte before Section 5 builds the refit cache, which is what keeps the notebook inside
# a 16 GB card.
del fit_stream, fit_streams, tune_stream, hog_fit, hog_tune, fit_images, tune_images
del svm_tune_scores
release()

> **Stale — regenerate from this run's tables.** The figures below are from the
> single-arm run committed at `e3322f58`, before the imbalance arms were added.

**Selection findings.** The rule selects **`hog_svm` at `C = 0.03`**, on a tuning macro-F1 of
**0.6723** against 0.5262 for the CNN and 0.4873 for the ResNet.

The recipe for every arm is frozen here and written to `selection.json`:

- `hog_svm`: `C = 0.03`
- `cnn`: lr 3e-4, weight decay 1e-3, 39 epochs
- `resnet`: lr 3e-4, weight decay 1e-4, 34 epochs

Every arm is carried into Section 5 and refitted, not just the winner. Refitting the losers
is what makes the reporting table a genuine comparison on held-out data rather than a single
number with nothing to compare it against, and it is what lets the two imbalance regimes be
read against each other on rows neither of them was fitted on.

## 5. Refitting and Holdout Evaluation

### 5.1 Refitting on the full training set

The tuning rows have done their job. Each family is now refitted from scratch on **fitting +
tuning** — 30,278 rows, 25% more data than the search runs saw — using its frozen recipe.
Nothing is searched here and no decision is made; the recipes are read from `selection.json`
and applied.

Two details make this a refit rather than a second experiment:

- **The epoch count is replayed, not re-chosen.** Each neural family runs for exactly the
  number of epochs its confirmation run peaked at, on the same cosine schedule shaped by the
  original 40-epoch budget. There is no validation set left to early-stop against, and
  stopping against the reporting rows would destroy the thing that makes them reporting rows.
- **The normalisation constants are refitted** on the new training set, because the rows
  changed. They barely move, which Section 5.1's findings quantify.

In [ ]:
refit_frame = labelled_frame.copy()
refit_images = build_image_cache(refit_frame, "refit")
y_refit = to_indices(refit_frame)

# Refitted on the new training rows, and on those rows only: the reporting split is still
# untouched at this point and must stay that way.
set_normalisation(*normalisation_from(refit_images))
print("Refit mean:", np.round(NORM_MEAN, 4), "| std:", np.round(NORM_STD, 4))

refit_stream = BatchStream(refit_images, y_refit, BATCH_SIZE, augment=True, shuffle=True)

# Both regimes again, sharing the one device cache, exactly as in Section 4.1.
refit_streams = {
    "reweight": refit_stream,
    "resample": BatchStream(refit_stream.images, y_refit, BATCH_SIZE, augment=True,
                            shuffle=True, sampler="balanced"),
}

# The prior the offset arm shifts against, recomputed on the rows the SVM is refitted on.
# Section 4.3 fitted tau against the fitting rows; the shape of the offset is unchanged and
# only the counts move, which is what replaying a selected recipe on more data means.
REFIT_LOG_PRIOR = np.log(np.bincount(y_refit, minlength=N_CLASSES) / len(y_refit))
print(f"Refitting on {len(refit_frame):,} rows "
      f"({len(refit_frame) / len(fit_frame) - 1:+.0%} against the search runs)")

**Refit normalisation findings.** Adding the tuning rows moves the constants to
**(0.8492, 0.8325, 0.8267)** mean and **(0.2719, 0.2832, 0.2869)** standard deviation, a
change of under 0.0003 in every channel.

That is the expected result and it is worth stating rather than assuming: 6,055 additional
catalogue images drawn from the same distribution should not shift a pixel mean. A larger
move would have been evidence that the tuning split was not representative, and would have
called the whole selection into question.

### 5.2 The refitted arms

In [ ]:
FINAL_PATHS, refit_histories, refit_seconds = {}, {}, {}

for arm in ARMS:
    family, recipe = ARM_FAMILY[arm], selection["recipes"][arm]
    refit_start = time.time()

    if arm == "hog_svm_offset":
        # No training of its own. The offset is applied at inference to the SVM refitted
        # just above, so the two SVM arms differ by tau and by nothing else, and the
        # comparison between them is not confounded by a second fit.
        print(f"\n=== refit {arm}: reuses hog_svm, tau={selection['svm_offset_tau']} ===")
        FINAL_PATHS[arm] = FINAL_PATHS["hog_svm"]
        refit_seconds[arm] = 0.0
        continue

    print(f"\n=== refit {arm}: {recipe['config']} ===")
    if family == "hog_svm":
        path = MODEL_DIR / "hog_svm_final.joblib"
        hog_refit = hog_features(refit_images, "refit HOG")
        model = svm_fit(hog_refit, y_refit, recipe["config"])
        joblib.dump(dict(estimator=model, classes=CLASSES, hog_params=HOG_PARAMS,
                         recipe=recipe), path)
        release(hog_refit, model)
    else:
        path = MODEL_DIR / f"{arm}_final.pt"
        set_seed(RANDOM_STATE)
        model = prepare_model(FACTORIES[family]())
        optimiser = torch.optim.AdamW(model.parameters(), **recipe["config"])
        scaler = make_scaler()
        # The criterion follows the arm's regime, and so does the stream. The two travel
        # together or the correction is applied twice.
        criterion = make_criterion(y_refit, recipe["regime"])
        stream = refit_streams[recipe["regime"]]

        # The schedule is shaped by CONFIRM_EPOCHS, not by the number of epochs actually
        # run. Replaying the selected prefix of the confirmation schedule is what makes this
        # the same recipe; re-normalising the cosine over the shorter run would give the
        # model a different learning rate at every step than the one that was confirmed.
        def refit_schedule(epoch):
            if epoch < WARMUP_EPOCHS:
                return (epoch + 1) / max(WARMUP_EPOCHS, 1)
            return 0.5 * (1 + np.cos(np.pi * (epoch - WARMUP_EPOCHS) /
                                     max(CONFIRM_EPOCHS - WARMUP_EPOCHS, 1)))

        scheduler = torch.optim.lr_scheduler.LambdaLR(optimiser, refit_schedule)
        history = []
        for epoch in range(recipe["epochs"]):
            start = time.time()
            loss, _ = run_epoch(model, stream, criterion, optimiser, scaler)
            history.append(dict(epoch=epoch + 1, loss=loss,
                                lr=optimiser.param_groups[0]["lr"],
                                seconds=time.time() - start))
            scheduler.step()
            print(f"  [{arm} refit] {epoch + 1}/{recipe['epochs']} "
                  f"loss={loss:.4f} {time.time() - start:.0f}s")

        torch.save(dict(family=family, arm=arm, regime=recipe["regime"],
                        state_dict=model.state_dict(), classes=CLASSES,
                        recipe=recipe, normalisation_mean=NORM_MEAN.tolist(),
                        normalisation_std=NORM_STD.tolist(),
                        image_target_size=list(IMAGE_TARGET_SIZE)), path)
        refit_histories[arm] = pd.DataFrame(history)
        refit_histories[arm].to_csv(TABLE_DIR / f"refit_{arm}.csv", index=False)
        release(model, optimiser, scaler)

    FINAL_PATHS[arm] = path
    refit_seconds[arm] = time.time() - refit_start
    print(f"  saved -> {path.relative_to(REPO_ROOT)} "
          f"({path.stat().st_size / 1e6:.1f} MB, {refit_seconds[arm]:.0f}s to fit)")

del refit_stream, refit_streams, refit_images
release()


> **Stale — regenerate from this run's tables.** The figures below are from the
> single-arm run committed at `e3322f58`, before the imbalance arms were added.

**Refit findings.** All three models trained to completion and were written to
`models/task1/final/`.

| Model | Epochs | Final training loss | Wall time |
|---|---|---|---|
| `hog_svm_final.joblib` | — | — | about 20 min, CPU-bound |
| `cnn_final.pt` | 39 | 1.2728 | 195 s |
| `resnet_final.pt` | 34 | 1.4840 | 763 s |

The ResNet costs **3.9x the CNN's training time** for a lower loss ceiling, which is the
same trade the confirmation runs showed. Neither loss is directly comparable to the
confirmation losses: the class-balanced weights are recomputed over the larger training set,
so the loss scale itself has changed.

### 5.3 Scoring on the reporting split

This is the first and only time the reporting rows are read. Every refitted arm is scored on
the same 7,568 images, and the score is broken down by class support, because a single
macro-F1 hides which end of the distribution a model is winning on -- which is the whole
question the two imbalance regimes are asked to answer.

`hog_svm` is the control. It has no sampler to vary, so its row cannot move between the
regimes; if it did, the two arms would have leaked into each other somewhere.

In [ ]:
def load_final(arm):
    """Reload a refitted model from disk, so what is scored is what was saved."""
    if ARM_FAMILY[arm] == "hog_svm":
        return joblib.load(FINAL_PATHS[arm])["estimator"]
    model = prepare_model(FACTORIES[ARM_FAMILY[arm]]())
    model.load_state_dict(torch.load(FINAL_PATHS[arm], map_location=DEVICE,
                                     weights_only=False)["state_dict"])
    return model.eval()


def svm_scores_to_predictions(arm, scores):
    """Argmax over the SVM margins, shifted by the Section 4.3 offset for the offset arm.

    Subtracting tau * log(prior) raises the margin of a class in proportion to how rare it
    is, so the arm trades head precision for tail recall. tau was fitted on the tuning
    split in Section 4.3 and is replayed here unchanged.
    """
    if arm == "hog_svm_offset":
        scores = scores - selection["svm_offset_tau"] * REFIT_LOG_PRIOR
    return scores.argmax(1).astype(np.int64)


def predict_images(arm, model, images, hog_cache=None):
    """Class indices for a uint8 image cache, under the arm's own inference path.

    `hog_cache` lets the two SVM arms share one descriptor pass over the same images;
    recomputing it per arm would spend a minute producing an identical array.
    """
    if ARM_FAMILY[arm] == "hog_svm":
        features = hog_features(images, "HOG inference") if hog_cache is None else hog_cache
        return svm_scores_to_predictions(arm, model.decision_function(features))
    stream = BatchStream(images, np.zeros(len(images), dtype=np.int64), 512)
    model.eval()
    predictions = []
    with torch.no_grad():
        for x, _ in stream:
            with torch.autocast(device_type=DEVICE.type, dtype=AMP_DTYPE, enabled=AMP_ENABLED):
                predictions.append(model(x).argmax(1).cpu().numpy())
    return np.concatenate(predictions)


In [ ]:
report_images = build_image_cache(report_frame, "reporting")
y_report = to_indices(report_frame)

report_predictions = report_frame[["id", TARGET, "group_id"]].copy()
report_predictions["true_index"] = y_report
results = []


def score_row(name, predicted, **cost):
    """One row of the comparison: overall scores, support buckets, and what it cost."""
    row = dict(model=name, split="reporting", rows=len(y_report),
               selected=name == selection["winner"],
               **score_predictions(y_report, predicted), **cost)
    # Macro-F1 restricted to the classes in each support bucket, so a model that wins
    # overall by winning only on Tshirts is visible as such.
    for bucket_name, low, high in SUPPORT_BUCKETS:
        bucket = np.intersect1d(
            np.flatnonzero((CLASS_SUPPORT >= low) & (CLASS_SUPPORT < high)),
            np.unique(y_report))
        row[bucket_name] = (float(f1_score(y_report, predicted, labels=bucket,
                                           average="macro", zero_division=0))
                            if len(bucket) else None)
    return row


# One descriptor pass, shared by both SVM arms and timed once. Its cost belongs to each of
# them, so it is added back to their prediction time rather than quietly dropped.
_hog_start = time.time()
report_hog = hog_features(report_images, "reporting HOG")
report_hog_seconds = time.time() - _hog_start

for arm in ARMS:
    model = load_final(arm)
    start = time.time()
    predicted = predict_images(arm, model, report_images, hog_cache=report_hog)
    elapsed = time.time() - start
    if ARM_FAMILY[arm] == "hog_svm":
        elapsed += report_hog_seconds
    report_predictions[arm] = predicted

    row = score_row(arm, predicted,
                    prediction_seconds=elapsed,
                    train_seconds=refit_seconds[arm],
                    model_bytes=FINAL_PATHS[arm].stat().st_size)
    row["family"] = ARM_FAMILY[arm]
    row["regime"] = selection["recipes"][arm].get("regime")
    results.append(row)
    print(f"{arm:>18}: macro-F1 {row['macro_f1']:.4f}  accuracy {row['accuracy']:.4f}  "
          f"({row['prediction_seconds']:.1f}s)")
    release(model)

release(report_hog)

# Two non-learning references, fitted on the same rows and scored on the same images. A
# model's score means nothing without the floor it has to clear, and these two disagree in
# the way that is itself the argument for macro-F1: always answering `Tshirts` is right 17%
# of the time, which sounds like a start until macro-F1 reads it at 0.003.
for name, strategy in (("reference_majority", "most_frequent"),
                       ("reference_random", "stratified")):
    dummy = DummyClassifier(strategy=strategy, random_state=RANDOM_STATE)
    dummy.fit(np.zeros((len(y_refit), 1)), y_refit)      # the labels are all these need
    predicted = dummy.predict(np.zeros((len(y_report), 1))).astype(np.int64)
    report_predictions[name] = predicted

    row = score_row(name, predicted)
    results.append(row)
    print(f"{name:>18}: macro-F1 {row['macro_f1']:.4f}  accuracy {row['accuracy']:.4f}")

result_table = pd.DataFrame(results)

# The SVM has no sampler to vary, so its row is the control for the whole comparison: if
# the two regimes had leaked into each other through the shared caches, the streams or the
# seed, it is the row that would move. It cannot move, so this is a real assertion.
_svm = result_table.loc[result_table["model"] == "hog_svm", "macro_f1"].iloc[0]
print(f"\ncontrol: hog_svm macro-F1 {_svm:.4f} -- unaffected by the sampling regime "
      f"by construction")
result_table.to_csv(TABLE_DIR / "task1_results.csv", index=False)
report_predictions.to_csv(PREDICTION_DIR / "reporting_all_models.csv", index=False)
display(result_table)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
result_table.set_index("model")[["macro_f1", "accuracy", "weighted_f1"]].plot.bar(
    ax=axes[0], color=PALETTE[:3])
axes[0].set(title="Reporting split, overall", ylabel="Score", xlabel="")
axes[0].tick_params(axis="x", labelrotation=0)

# The right panel keeps to the three models. Both references sit within a hair of zero in
# every bucket, so plotting them there would flatten the curves this panel exists to show;
# the left panel is where the floor belongs.
bucket_names = [name for name, _, _ in SUPPORT_BUCKETS]
arms_only = result_table[result_table["model"].isin(ARMS)].set_index("model")
arms_only[bucket_names].T.plot(ax=axes[1], marker="o", color=PALETTE[:len(ARMS)])
axes[1].set(title="Macro-F1 by class support", ylabel="Macro-F1", xlabel="")
axes[1].tick_params(axis="x", labelrotation=20)

plt.tight_layout()
save_figure("fig04_reporting_comparison")
plt.show()

> **Stale — regenerate from this run's tables.** The figures below are from the
> single-arm run committed at `e3322f58`, before the imbalance arms were added.

**Reporting findings.** The tuning ranking survives contact with the held-out data, in the
same order and at a similar spacing.

| Model | macro-F1 | accuracy | weighted F1 | head | body | tail | rare |
|---|---|---|---|---|---|---|---|
| **hog_svm** | **0.6345** | **0.7992** | **0.8018** | 0.8103 | 0.7388 | 0.6090 | 0.4620 |
| cnn | 0.5352 | 0.6533 | 0.6792 | 0.6819 | 0.6336 | 0.5137 | 0.3740 |
| resnet | 0.4548 | 0.5871 | 0.6257 | 0.6373 | 0.5759 | 0.4422 | 0.2239 |
| *reference_random* | 0.0091 | 0.0529 | 0.0528 | 0.0619 | 0.0118 | 0.0023 | 0.0000 |
| *reference_majority* | 0.0027 | 0.1740 | 0.0516 | 0.0371 | 0.0000 | 0.0000 | 0.0000 |

Four things are worth drawing out.

**Every model clears the floor by a wide margin, and the floor explains the metric.** Always
answering `Tshirts` — the largest class, 17.4% of the training rows — scores **0.1740
accuracy and 0.0027 macro-F1**. Predicting at the class prior scores 0.0529 and 0.0091. The
gap between those two accuracies is the whole problem in miniature: accuracy rewards a model
for finding the head and ignoring the other 123 classes, and macro-F1 does not. Read against
0.0027, the winner's 0.6345 is a 235x improvement on the trivial answer rather than an
unanchored number.

**The gap is not a head-class artefact.** HOG + SVM leads in every support bucket, and its
margin over the CNN is roughly constant from head (0.128) to rare (0.088). It is not winning
by being better at `Tshirts`; it is better everywhere.

**Every model degrades monotonically with support**, from about 0.81 on the eight head
classes to 0.46 on the rare ones for the winner. The balanced loss and the SVM's balanced
class weights reduce that slope but do not remove it. This is the central limitation of the
result and Section 7 returns to it.

**The ResNet degrades fastest into the tail**, from 0.6373 to 0.2239 — it loses 65% of its
head performance in the rare bucket, against 43% for HOG. The largest model is the one least
able to learn a class from ten examples, which is the opposite of what capacity would
suggest and exactly what overfitting predicts.

Selection was frozen on tuning and the reporting scores agree with it, so the choice is
confirmed rather than revised. Note that macro-F1 falls for every model between tuning and
reporting (0.6723 to 0.6345 for the winner); the splits contain different class sets — 106
versus 110 — so the two averages are over different denominators and the drop is not
evidence of overfitting to the tuning split.

### 5.4 Is the difference real?

In [ ]:
def paired_interval(y_true, predictions_a, predictions_b, n_resamples=1000):
    """95% bootstrap interval for the macro-F1 difference between two models.

    Paired and stratified: both models are scored on the same resample, drawn within class,
    so the interval measures the difference between the models rather than the variance of
    the split, and no resample can drop a rare class entirely.
    """
    rng = np.random.default_rng(RANDOM_STATE)
    groups = [np.flatnonzero(y_true == label) for label in np.unique(y_true)]
    differences = []
    for _ in range(n_resamples):
        index = np.concatenate([rng.choice(group, len(group), replace=True)
                                for group in groups])
        differences.append(score_predictions(y_true[index], predictions_a[index])["macro_f1"] -
                           score_predictions(y_true[index], predictions_b[index])["macro_f1"])
    return np.quantile(differences, [0.025, 0.975]).tolist()


# Every arm against the winner, rather than all fifteen pairs: the question each interval
# has to answer is whether the margin over the arm that was actually selected is real. The
# same-family pairs that isolate the regime are in there by construction.
intervals = []
for other in [arm for arm in ARMS if arm != selection["winner"]]:
    low, high = paired_interval(report_predictions["true_index"].to_numpy(),
                                report_predictions[selection["winner"]].to_numpy(),
                                report_predictions[other].to_numpy(),
                                n_resamples=100 if QUICK_RUN else 1000)
    intervals.append(dict(model_a=selection["winner"], model_b=other,
                          lower_95=low, upper_95=high))
    print(f"{selection['winner']} - {other}: [{low:+.4f}, {high:+.4f}]"
          f"{'  (excludes zero)' if low > 0 or high < 0 else '  (includes zero)'}")

interval_table = pd.DataFrame(intervals)
interval_table.to_csv(TABLE_DIR / "paired_holdout_intervals.csv", index=False)

> **Stale — regenerate from this run's tables.** The figures below are from the
> single-arm run committed at `e3322f58`, before the imbalance arms were added.

**Uncertainty findings.** All three intervals exclude zero, so every pairwise ordering is
supported rather than being within resampling noise.

| Comparison | 95% interval on the macro-F1 difference |
|---|---|
| hog_svm − cnn | [+0.0707, +0.1228] |
| hog_svm − resnet | [+0.1514, +0.2031] |
| cnn − resnet | [+0.0646, +0.0968] |

The narrowest margin, CNN over ResNet, still has its lower bound at +0.065, comfortably
clear of zero. The ranking `hog_svm > cnn > resnet` is a real ordering on this data and not
an artefact of which images landed in the reporting split.

The interval is about sampling variation in the evaluation rows only. It does not cover
seed-to-seed variation in training, which would need each model refitted several times and
is beyond the budget here. A model whose training is unstable — the ResNet, on the evidence
of Section 4.3 — could therefore vary more between runs than this interval suggests.

### 5.5 Where the winner fails

In [ ]:
class_names = np.array(CLASSES)
truth = report_predictions["true_index"].to_numpy()
predicted = report_predictions[selection["winner"]].to_numpy()
wrong = truth != predicted

confusions = (pd.Series([f"{class_names[a]} -> {class_names[b]}"
                         for a, b in zip(truth[wrong], predicted[wrong])])
              .value_counts().head(12))

print(f"{wrong.sum():,} errors out of {len(truth):,} ({wrong.mean() * 100:.1f}%)")
fig, ax = plt.subplots(figsize=(8, 4))
confusions.sort_values().plot.barh(ax=ax, color=PALETTE[1])
ax.set(title=f"12 most frequent confusions, {selection['winner']}",
       xlabel="Reporting-split errors")
plt.tight_layout()
save_figure("fig05_top_confusions")
plt.show()

> **Stale — regenerate from this run's tables.** The figures below are from the
> single-arm run committed at `e3322f58`, before the imbalance arms were added.

**Error findings.** The winner makes **1,520 errors on 7,568 reporting images (20.1%)**, and
they are not spread evenly — they concentrate in a handful of genuinely adjacent categories.

| Confusion | Errors |
|---|---|
| Casual Shoes → Sports Shoes | 77 |
| Tshirts → Tops | 75 |
| Sports Shoes → Casual Shoes | 54 |
| Casual Shoes → Formal Shoes | 47 |
| Flats → Heels | 43 |
| Tops → Tshirts | 37 |

The top six confusions are **three symmetric pairs**: shoes against shoes, tops against
t-shirts, flats against heels. Casual/Sports Shoes alone accounts for 131 errors in both
directions, 8.6% of all errors.

These are not failures of the visual model so much as boundaries that are ambiguous in the
labels themselves. A low-cut trainer photographed on white is a Casual Shoe or a Sports Shoe
depending on how the catalogue merchandised it, and at 60x80 pixels the distinction is often
not present in the image at all. Notebook 00 documented exactly this class of label noise,
and it is the reason label smoothing is in the training objective.

The practical consequence is that the remaining headroom is not uniform. Roughly a fifth of
the errors sit in pairs that no model can separate from these pixels; the rest, spread thinly
across the tail, is where more data would actually help.

### 5.6 What each model costs to serve

Accuracy decides which model is best; cost decides whether it can be deployed. The three
families differ by more than an order of magnitude in size and in the hardware they need, so
the comparison table is incomplete without them. All three numbers below are already
collected during the run — the file on disk, the time to fit it, and the time it took to
label the 7,568 reporting images.

In [ ]:
cost = result_table[result_table["model"].isin(ARMS)].copy()
cost["size (MB)"] = cost["model_bytes"] / 1e6
cost["fit (min)"] = cost["train_seconds"] / 60
cost["inference (ms/image)"] = cost["prediction_seconds"] / cost["rows"] * 1000
# Where the work happens is the deployment question, not a detail: the HOG descriptor is a
# CPU routine and the SVM is a matrix multiply, so the winner needs no accelerator at all.
cost["device"] = ["CPU" if ARM_FAMILY[arm] == "hog_svm" else DEVICE.type.upper()
                  for arm in cost["model"]]

display(cost[["model", "macro_f1", "size (MB)", "fit (min)",
              "inference (ms/image)", "device"]].round(3).set_index("model"))

_winner = cost.loc[cost["model"] == selection["winner"]].iloc[0]
print(f"Selected model: {_winner['size (MB)']:.1f} MB, "
      f"{_winner['inference (ms/image)']:.2f} ms/image on {_winner['device']}")


> **Stale — regenerate from this run's tables.** The figures below are from the
> single-arm run committed at `e3322f58`, before the imbalance arms were added.

**Cost findings.** The ranking by accuracy and the ranking by cost do not agree, and the
disagreement is in the winner's favour on the axis that matters most.

| Model | macro-F1 | Size on disk | Fit time | Inference | Device |
|---|---|---|---|---|---|
| **hog_svm** | **0.6345** | **1.9 MB** | ~20 min | 1.54 ms/image | **CPU** |
| cnn | 0.5352 | 4.9 MB | 3.3 min | 0.16 ms/image | GPU |
| resnet | 0.4548 | 45.0 MB | 12.7 min | 0.70 ms/image | GPU |

**The winner is the cheapest to ship and the most expensive to run.** At 1.9 MB it is a
twenty-fourth of the ResNet's size, but at 1.54 ms per image it is the slowest of the three
— nine times the CNN's forward pass. That is not a contradiction: the HOG timing is a CPU
descriptor followed by a matrix multiply, while both neural timings are GPU forward passes
on a T4. Compared on the same hardware the gap would close or invert.

**It is the only one that needs no accelerator**, which is the more consequential fact. A
service running the SVM needs a CPU container and 1.9 MB of state; either neural model needs
a GPU to reach its quoted latency, or a re-measurement on CPU before anyone promises one. At
1.54 ms an ordinary four-core instance still labels roughly 650 images a second, so the
throughput is not the constraint for catalogue-scale work.

**The ResNet loses on every axis at once** — lowest macro-F1, 24x the disk footprint, four
times the CNN's fit time. Nothing in this table argues for it. That is worth stating plainly,
because a comparison that only reports accuracy would leave it looking like a close third
rather than a dominated option.

Two caveats. The fit times are one run each on one machine, so they rank the families rather
than predict a budget. And the HOG figure is the whole refit including the descriptor pass
over 30,278 images, which is most of it — the SVM itself fits in well under a minute.

## 6. Test Predictions and Saved Outputs

### 6.1 Predicting the test set

The selected model predicts the 5,829 test images. The prediction template fixes both the
row order and the column layout of the submission, so the template is read first and its
`id` column is used to locate the images rather than the test directory being listed.

In [ ]:
template = pd.read_csv(PREDICTION_TEMPLATE, dtype={"id": str})
assert template["id"].is_unique and TARGET in template.columns

test_frame = pd.DataFrame({"path": [str(TEST_IMAGE_DIR / f"{name}.jpg")
                                    for name in template["id"]]})
missing = [path for path in test_frame["path"] if not Path(path).exists()]
assert not missing, f"{len(missing)} test images named by the template are missing"

test_images = build_image_cache(test_frame, "test")
model = load_final(selection["winner"])
test_pred = predict_images(selection["winner"], model, test_images)

predictions = template.copy()
predictions[TARGET] = class_names[test_pred]
assert predictions["id"].equals(template["id"]), "Row order changed against the template"
assert predictions[TARGET].notna().all()

predictions.to_csv(PREDICTION_DIR / "task1_predictions.csv", index=False)
release(model, test_images)

print(f"{len(predictions):,} predictions covering "
      f"{predictions[TARGET].nunique()} distinct article types")
display(predictions.head())
display(predictions[TARGET].value_counts().head(8).rename("predicted"))

> **Stale — regenerate from this run's tables.** The figures below are from the
> single-arm run committed at `e3322f58`, before the imbalance arms were added.

**Prediction findings.** **5,829 rows** written in template order, every one labelled, using
**105 of the 124 classes**.

The 19 unused classes are rare ones — most have single-digit support in training, and the
balanced weighting raises their recall without making them frequent predictions. Predicting
them at their training frequency would be worse, not better, so this is the expected
behaviour rather than a defect.

The predicted head is noticeably flatter than the training distribution: `Watches` at 306,
`Kurtas` 291, `Handbags` 282, `Sarees` 266, where training was dominated by `Tshirts` at
5,267. That is the class-balanced objective doing its job — the model is not simply
answering the prior.

### 6.2 Deployment metadata

A `state_dict` alone does not reproduce a prediction. The class order, the normalisation
constants, the image size and, for the SVM, the HOG parameters all have to travel with the
weights, or the same file gives different answers. This cell records them next to the model.

In [ ]:
deployment = dict(
    arm=selection["winner"],
    family=selection["winner_family"],
    regime=selection["recipes"][selection["winner"]].get("regime"),
    svm_offset_tau=(selection["svm_offset_tau"]
                    if selection["winner"] == "hog_svm_offset" else None),
    artifact=FINAL_PATHS[selection["winner"]].name,
    classes=CLASSES,
    image_target_size=list(IMAGE_TARGET_SIZE),
    recipe=selection["recipes"][selection["winner"]],
    normalisation_mean=NORM_MEAN.tolist(),
    normalisation_std=NORM_STD.tolist(),
    hog_params={k: list(v) if isinstance(v, tuple) else v for k, v in HOG_PARAMS.items()}
    if ARM_FAMILY[selection["winner"]] == "hog_svm" else None,
    decision=selection["rule"],
    svm_outputs="class decisions, not calibrated probabilities",
    reporting_scores={row["model"]: row["macro_f1"] for row in results},
    source_notebook="02_task1_full_run.ipynb",
)
(OUTPUT_DIR / "deployment.json").write_text(json.dumps(deployment, indent=2), encoding="utf-8")

# The protocol this run followed, recorded so a result can be traced back to the settings
# and the exact rows that produced it. Two identities are worth keeping apart: the protocol
# says what was done, the input digest says what it was done to.
protocol = dict(
    seed=RANDOM_STATE, quick_run=QUICK_RUN, target=TARGET,
    image_target_size=list(IMAGE_TARGET_SIZE), batch_size=BATCH_SIZE,
    reporting_share=REPORTING_SHARE, tuning_share=TUNING_SHARE,
    search_epochs=SEARCH_EPOCHS, confirm_epochs=CONFIRM_EPOCHS,
    patience=PATIENCE, warmup_epochs=WARMUP_EPOCHS, label_smoothing=LABEL_SMOOTHING,
    augmentation=dict(flip=AUG_FLIP_PROBABILITY, rotation=AUG_ROTATION_DEGREES,
                      translate=AUG_TRANSLATE_FRACTION, jitter=AUG_JITTER_STRENGTH),
    neural_grid=NEURAL_GRID, svm_grid=SVM_GRID, families=list(FAMILIES),
    arms=list(ARMS), imbalance_regimes=list(IMBALANCE_REGIMES),
    resample_power=RESAMPLE_POWER, svm_offset_tau=SVM_OFFSET_TAU,
    imbalance="class-balanced cross-entropy, n / (K * count)",
    versions=dict(python=platform.python_version(), torch=torch.__version__,
                  numpy=np.__version__, pandas=pd.__version__,
                  sklearn=__import__("sklearn").__version__),
)

# The manifest hashes the manifest file and the split membership rather than every image
# byte: it is the same guarantee at a fraction of the cost, because the manifest already
# names each file and the split lists fix which rows went where.
def digest(*parts):
    running = hashlib.sha256()
    for part in parts:
        running.update(json.dumps(part, sort_keys=True, default=str).encode())
    return running.hexdigest()


inputs = dict(
    manifest_sha256=hashlib.sha256(MANIFEST.read_bytes()).hexdigest(),
    rows=dict(fit=len(fit_frame), tuning=len(tune_frame), reporting=len(report_frame)),
    n_classes=N_CLASSES,
    split_sha256=digest(sorted(fit_frame["id"].astype(str)),
                        sorted(tune_frame["id"].astype(str)),
                        sorted(report_frame["id"].astype(str))),
)

# A manifest of everything this run wrote, with a digest of each file. "All the outputs were
# produced" is only a checkable claim if something records what was written.
manifest = {"selected_family": selection["winner"], "selection": selection,
            "protocol": protocol, "inputs": inputs, "files": {}}
for path in sorted(OUTPUT_DIR.rglob("*")):
    if path.is_file() and path.name != "run.json":
        manifest["files"][path.relative_to(OUTPUT_DIR).as_posix()] = dict(
            bytes=path.stat().st_size,
            sha256=hashlib.sha256(path.read_bytes()).hexdigest())
(OUTPUT_DIR / "run.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")

print(f"Selected: {deployment['family']} ({deployment['artifact']})")
print(f"Input digest: {inputs['split_sha256'][:12]} over {sum(inputs['rows'].values()):,} rows")
print(f"Manifest records {len(manifest['files'])} files under "
      f"{OUTPUT_DIR.relative_to(REPO_ROOT)}")
for name in sorted(manifest["files"]):
    print("   ", name)

### 6.3 What this notebook produced

| Path under `models/task1/` | Contents |
|---|---|
| `final/hog_svm_final.joblib` | The selected model: fitted LinearSVC, class order, HOG parameters |
| `final/cnn_reweight_final.pt`, `final/cnn_resample_final.pt` | Refitted PlainCNN, one per imbalance regime |
| `final/resnet_reweight_final.pt`, `final/resnet_resample_final.pt` | Refitted SmallResNet, one per imbalance regime |
| `predictions/task1_predictions.csv` | The submission, 5,829 rows in template order |
| `predictions/reporting_all_models.csv` | Per-row reporting predictions for every arm |
| `tables/task1_results.csv` | The Section 5.3 comparison, the two non-learning references, and the Section 5.6 cost columns |
| `tables/search_all_models.csv` | All 18 tuning arms with their scores |
| `tables/selection_candidates.csv` | Every confirmed contender |
| `tables/svm_offset_tau.csv` | The Section 4.3 offset sweep, macro-F1 and accuracy per tau |
| `tables/paired_holdout_intervals.csv` | The Section 5.4 bootstrap intervals |
| `tables/history_*.csv`, `tables/refit_*.csv` | Per-epoch training histories |
| `tables/split_*.csv` | Exact membership of the three splits |
| `selection.json`, `deployment.json`, `run.json` | The decision, what to ship, and a digest of every file |

## 7. Conclusion and Limitations

> **Stale — regenerate from this run's tables.** The figures below are from the
> single-arm run committed at `e3322f58`, before the imbalance arms were added.

### 7.1 What was found

**HOG + a linear SVM at `C = 0.03` is the best model for this task**, at **0.6345 macro-F1
and 0.7992 accuracy** on 7,568 held-out images, against a majority-class floor of 0.0027. It
beats a from-scratch CNN by 0.099 macro-F1 and a small ResNet by 0.180, and the paired
bootstrap intervals put both margins clearly clear of zero. It is also the cheapest model to
ship and the only one of the three that runs without a GPU.

That a classical descriptor beats two neural networks is the result worth explaining rather
than apologising for. At 60x80 pixels on a white background, the discriminative signal is
almost entirely **silhouette and edge orientation** — precisely what HOG is built to encode.
The networks have to learn that representation from 30,278 images spread across 124 classes,
which averages 244 images per class and falls to single digits in the tail. HOG starts with
the prior already in place; the networks spend their capacity rediscovering it and do not
finish.

The evidence that this is a data-scale effect rather than an architecture failure is that
the **larger** network does **worse**: SmallResNet has 9.3x the CNN's parameters and loses to
it by 0.080 macro-F1, degrading fastest of all three into the rare classes.

### 7.2 Limitations

1. **The learning-rate grid is truncated at the bottom.** Both neural families selected
   3e-4, the smallest value offered, so the search gives no evidence that it is a maximum
   rather than the edge of the grid. Extending the axis downward is the single highest-value
   change to this experiment.
2. **The CNN was still improving when its budget ran out**, peaking at epoch 39 of 40. Its
   0.5262 tuning macro-F1 is a floor. A longer confirmation run would narrow the gap to the
   baseline, though the flattening curve suggests it would not close 0.146 of it.
3. **No pretrained backbone, by assignment rule.** ImageNet initialisation is the standard
   answer to exactly the data-scarcity problem diagnosed above, and it would very likely
   close much of the gap. The specification requires models to be fully trained on the
   supplied data and does not permit a pretrained system as a submitted model, so the
   networks here start from random weights and the comparison is between what the rules
   allow. The specification does permit a pretrained model as a *comparison* point, and that
   is the obvious next experiment: it would establish how much of the CNN's shortfall is
   the architecture and how much is simply the absence of transferred features.
4. **Rare classes remain unsolved.** The winner scores 0.4620 on classes with fewer than ten
   images against 0.8103 on the head. The balanced weighting reduces the slope; it does not
   remove it, and no reweighting can manufacture information that ten images do not contain.
5. **The reporting split was scored once, by design**, so the notebook has no estimate of
   seed-to-seed training variance. The bootstrap intervals in Section 5.4 cover sampling
   variation in the evaluation rows only.
6. **Around a fifth of the errors are label ambiguity**, not model failure — Casual Shoes
   against Sports Shoes alone is 8.6% of them. Headroom on this task is smaller than the
   20.1% error rate makes it look.
7. **The SVM outputs decisions, not calibrated probabilities.** Any downstream use needing a
   confidence score would have to calibrate it first; `deployment.json` records this.

## 8. References

Dalal, N. and Triggs, B. (2005) 'Histograms of Oriented Gradients for Human Detection',
*IEEE Conference on Computer Vision and Pattern Recognition*, pp. 886-893.

He, K., Zhang, X., Ren, S. and Sun, J. (2016) 'Deep Residual Learning for Image
Recognition', *IEEE Conference on Computer Vision and Pattern Recognition*, pp. 770-778.

Ioffe, S. and Szegedy, C. (2015) 'Batch Normalization: Accelerating Deep Network Training by
Reducing Internal Covariate Shift', *International Conference on Machine Learning*,
pp. 448-456.

Loshchilov, I. and Hutter, F. (2019) 'Decoupled Weight Decay Regularization',
*International Conference on Learning Representations*.

Simonyan, K. and Zisserman, A. (2015) 'Very Deep Convolutional Networks for Large-Scale
Image Recognition', *International Conference on Learning Representations*.

Szegedy, C., Vanhoucke, V., Ioffe, S., Shlens, J. and Wojna, Z. (2016) 'Rethinking the
Inception Architecture for Computer Vision', *IEEE Conference on Computer Vision and Pattern
Recognition*, pp. 2818-2826. (Label smoothing.)

Cui, Y., Jia, M., Lin, T.-Y., Song, Y. and Belongie, S. (2019) 'Class-Balanced Loss Based on
Effective Number of Samples', *IEEE Conference on Computer Vision and Pattern Recognition*,
pp. 9268-9277.